# LoRA training: `t2a_lora` (4x L4)
Generated by `kaggle/train/make_train_notebook.py` from `kaggle/train/trainkit/` (edit there, not here).
Recipe: NF4 from the bf16 QAT weights, LoRA r16 on 410 linears, FSDP over 4 ranks, exact full backprop up to 28k
tokens with the long-context fixes (section 4), loss weighted per token and normalised per optimizer step.
Data: `rs4`.

## 1. Settings
Everything the run needs is in `train_cfg.json`; the budget clock starts here.

In [ ]:
import json, os, sys, time
for d in ("/kaggle/working/evalkit", "/kaggle/working/trainkit"):
    os.makedirs(d, exist_ok=True)
TRAIN_CFG = {'name': 't2a_lora', 'lr': 0.0001, 'max_windows': None, 'budget_h': 10.5, 'smoke': False, 'epochs': 1.0, 'grad_accum': 2, 'max_len': 28672, 'save_every': 24, 'val_every': 24, 'prefix_probe': False, 'lc_probe': False, 'lc_tests': 'UABC', 'vablate': False, 'corr': 'exclude', 'corr_share': 0.0}
TRAIN_CFG["t0"] = time.time()
open("/kaggle/working/train_cfg.json", "w").write(json.dumps(TRAIN_CFG, indent=1))
TRAIN_CFG

## 2. Embedded inputs
Tokenized windows (unless attached as a dataset) and, for the smoke model, the real config.

In [ ]:
import base64, pathlib
DATA_B64 = ''
if DATA_B64:
    pathlib.Path("/kaggle/working/train_data.npz").write_bytes(base64.b64decode(DATA_B64))
G4CFG = ''
if G4CFG:
    pathlib.Path("/kaggle/working/g4cfg.json").write_text(G4CFG)
print("data embedded:", bool(DATA_B64), "| smoke config:", bool(G4CFG))

## 3. Environment modules (from evalkit)

### `evalkit/__init__.py`
evalkit: scorer-faithful local evaluation of a submission on Kaggle 4x L4 (harness swegemma 0.2.11 / adk-submission 0.2.13).

In [ ]:
%%writefile /kaggle/working/evalkit/__init__.py
"""
evalkit: scorer-faithful local evaluation of a submission on Kaggle 4x L4 (harness swegemma 0.2.11 / adk-submission 0.2.13).

Modules, in the order the notebook uses them:
  config      Settings (one JSON file shared by every step) and fixed paths
  util        run.log logger + shell helper
  envsetup    ScorerEnv: restore the Py3.12 scorer venv, upgrade changed wheelhouse wheels, add test deps (stdlib only:
              runs in the notebook's own Python 3.13 kernel)
  inputs      Inputs: competition / dev-set tasks, snapshots, served model
  submission  Submission: unpack the submission under test, declared model, adapters, eval budget
  server      ScorerServer: vLLM with the scorer's settings + model registry; offline sandbox
  runner      stratified task order + ChunkedEvaluation (official Evaluator, sequential, rep-major repeats)
  report      Report: merge chunks per rep, pass@k summary, trace analysis, package
  main        orchestration of everything after the env step (runs inside the venv)
Source of truth: kaggle/eval_l4/evalkit/. make_eval_notebook.py turns it into the competition notebook.
"""

### `evalkit/config.py`
Run settings and fixed paths. Stdlib only, so the env step can import it outside the venv.

In [ ]:
%%writefile /kaggle/working/evalkit/config.py
"""Run settings and fixed paths. Stdlib only, so the env step can import it outside the venv."""
import json
import pathlib
import time
from dataclasses import asdict, dataclass, field

WORK = pathlib.Path("/kaggle/working")
INPUT = pathlib.Path("/kaggle/input")
VENV = pathlib.Path("/tmp/scorer_host")
TARGET_MODEL = "gemma-4-31b-it-qat-w4a16-ct"
CFG_FILE = WORK / "eval_cfg.json"
SUBMISSION_ZIP = WORK / "submission_src.zip"
ANALYZER = WORK / "trace_analyzer.py"
SANDBOX_ENV = WORK / "sandbox_env.json"   # task-sandbox interpreter + site, written by envsetup
SCORER_SITE = pathlib.Path("/tmp/scorer_site")
SANDBOX_PIP = pathlib.Path("/tmp/sandbox_pip")
VALID_TASKS = WORK / "valid_tasks.json"       # from the preflight sweep; embedded by make_eval_notebook.py


@dataclass
class Settings:
    name: str = "v7_noedit"           # submissions/<name>; label of every output
    task_ids: list | None = None      # None -> all tasks, interleaved by repo (any prefix is a stratified sample)
    max_tasks: int | None = None      # cap the task list (shakedown runs)
    seed: int = 0
    chunk_size: int = 8
    chunk_timeout_min: float = 60     # floor; the runner raises it to chunk_size x (task max_time + 3 min) (Oct 19)
    session_budget_h: float = 10.5    # stop scheduling chunks after this; the session cap is 12 h
    est_task_min: float = 5.5         # 4.5 min agent budget + setup/verification
    concurrency: int = 1              # tasks in flight per Evaluator call (1 = scorer; >1 only for data generation)
    max_time_minutes: float | None = None  # override the submission's per-task agent time cap (data generation only)
    shard: int = 0                    # factory rollouts: this kernel runs tasks[shard::n_shards]
    n_shards: int = 1
    repeats: int = 1                  # >1: run the list again (rep-major) -> pass@k, trajectories for rejection sampling
    devset: bool = False              # True -> renamed dev set (kernel output gemma4-devset-build)
    valid_only: bool = False          # True -> only tasks in valid_tasks.json (gold passes, empty fails; preflight sweep)
    sweep: bool = False               # preflight: gold + empty over ALL tasks -> valid_tasks.json (else 7-task smoke)
    gate_adapters: list | None = None  # behaviour gate: adapter names to compare with base (evalkit/gate.py)
    gate_samples: int = 4             # behaviour gate: samples per decision point
    t0: float = field(default_factory=time.time)  # notebook start; budget counts from here across processes

    @property
    def label(self):
        return f"eval_{self.name}" + ("_dev" if self.devset else "")

    def hours_left(self):
        return self.session_budget_h - (time.time() - self.t0) / 3600

    def save(self, path=CFG_FILE):
        path.write_text(json.dumps(asdict(self), indent=1))

    @classmethod
    def load(cls, path=CFG_FILE):
        return cls(**json.loads(path.read_text()))

### `evalkit/util.py`
Logger that writes to stdout and /kaggle/working/run.log, plus a logged shell helper. Stdlib only.

In [ ]:
%%writefile /kaggle/working/evalkit/util.py
"""Logger that writes to stdout and /kaggle/working/run.log, plus a logged shell helper. Stdlib only."""
import glob
import subprocess
import time

from .config import WORK


class Log:
    def __init__(self, path=WORK / "run.log"):
        self.f = open(path, "a", buffering=1)

    def __call__(self, *a):
        s = time.strftime("%H:%M:%S ") + " ".join(str(x) for x in a)
        print(s, flush=True)
        self.f.write(s + "\n")

    def sh(self, cmd, timeout=None, check=False):
        self("$", cmd[:220])
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        out = ((r.stdout or "")[-1500:] + ("\n" + r.stderr[-1500:] if (r.stderr or "").strip() else "")).strip()
        if out:
            self(out)
        if check and r.returncode != 0:
            raise SystemExit(f"FAILED rc={r.returncode}: {cmd[:120]}")
        return r


log = Log()


def first(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

### `evalkit/envsetup.py`
ScorerEnv: the scorer's Python environment, offline. Runs in the notebook kernel (Python 3.13), stdlib only.

In [ ]:
%%writefile /kaggle/working/evalkit/envsetup.py
"""
ScorerEnv: the scorer's Python environment, offline. Runs in the notebook kernel (Python 3.13), stdlib only.

The wheelhouse is cp312-only and not a consistent dependency set, so the venv is built once on a CPU kernel with
Internet (kaggle/env/build_scorer_env.py -> scorer_env.tgz). Here it is restored, then every wheelhouse wheel whose
version differs from the venv (Oct 9: swegemma 0.2.11, adk-submission 0.2.13; their requirements are unchanged)
is installed with --no-deps, plus the test-only deps of the fastapi tasks (dataset test-deps-wheels).
"""
import json
import os
import pathlib
import re
import sys
import time

from .config import INPUT, SANDBOX_ENV, SANDBOX_PIP, SCORER_SITE, VENV
from .util import first, log

TEST_DEPS = re.compile(r"(inline_snapshot|dirty_equals|asttokens|executing)-")


def canon(name):
    return re.sub(r"[-_.]+", "-", name).lower()


class ScorerEnv:
    def __init__(self, venv=VENV, n_gpus=4):
        self.venv, self.n_gpus = pathlib.Path(venv), n_gpus
        self.python = self.venv / "bin" / "python"

    def check_gpus(self):
        log.sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; nproc; free -g | head -2")
        gpus = [g for g in log.sh("nvidia-smi --query-gpu=name --format=csv,noheader").stdout.split("\n") if g.strip()]
        if len(gpus) < self.n_gpus:  # don't burn quota on the wrong machine
            raise SystemExit(f"need GPU L4 x{self.n_gpus}, got {gpus}")

    def restore(self):
        tgz = first(f"{INPUT}/**/scorer_env.tgz")
        assert tgz, "attach the output of kernel gemma4-scorer-env (kernel_sources)"
        log.sh(f"tar -xzf {tgz} -C /", timeout=1800, check=True)

    def installed(self):
        out = log.sh(f"{self.python} -c 'import importlib.metadata as m, json; "
                     "print(json.dumps({d.metadata[\"Name\"]: d.version for d in m.distributions()}))'",
                     check=True).stdout.strip().splitlines()[-1]
        return {canon(k): v.split("+")[0] for k, v in json.loads(out).items()}

    def stale_wheels(self):
        swegemma = first(f"{INPUT}/**/swegemma-*.whl")
        assert swegemma, "attach dataset metric/gemma-4-developer-agent-wheelhouse"
        have, stale = self.installed(), []
        for w in sorted(pathlib.Path(swegemma).parent.glob("*.whl")):
            if "cutlass" in w.name.lower():
                continue
            n, v = w.name.split("-")[:2]
            v = re.sub(r"\+?cu\d+$", "", v)
            if have.get(canon(n)) != v:
                log(f"wheelhouse {n} {v} != venv {have.get(canon(n))}")
                stale.append(w)
        return stale

    def test_dep_wheels(self):
        found = [pathlib.Path(p) for p in sorted(map(str, INPUT.rglob("*.whl"))) if TEST_DEPS.match(pathlib.Path(p).name)]
        if not found:
            log("WARNING: dataset test-deps-wheels not attached -> ~32 fastapi tasks will be unverifiable")
        return found

    def install(self, wheels):
        stage = pathlib.Path("/tmp/wh_stage")
        stage.mkdir(exist_ok=True)
        for w in wheels:  # pip rejects 'cu128' without '+' in a wheel's version
            name = w.name.replace("cu128", "+cu128") if ("cu128" in w.name and "+" not in w.name) else w.name
            if not (stage / name).exists():
                os.symlink(w, stage / name)
        whls = sorted(str(p) for p in stage.glob("*.whl"))
        if not whls:
            return
        r = log.sh(f"{sys.executable} -m pip --python {self.python} install --no-deps --no-index -q " + " ".join(whls),
                   timeout=1800)
        if r.returncode != 0:  # fallback: wheels are zip archives, unpack into site-packages
            site = log.sh(f"{self.python} -c 'import site; print(site.getsitepackages()[0])'").stdout.strip()
            for w in whls:
                log.sh(f"cd {site} && unzip -oq {w}", check=True)

    def ensure_pip(self):
        """uv venvs have no pip; the sandbox's per-task venvs need `python -m pip` (found via _host_env.pth) to
        install task packages. pip is pure Python, so copy the notebook Python's own pip into the venv (offline)."""
        if log.sh(f"{self.python} -m pip --version").returncode == 0:
            return
        import pip
        src = pathlib.Path(pip.__file__).parent
        site = pathlib.Path(log.sh(f"{self.python} -c 'import site; print(site.getsitepackages()[0])'",
                                   check=True).stdout.strip())
        log.sh(f"cp -r {src} {site}/ && cp -r {src.parent}/pip-*.dist-info {site}/", check=True)
        log.sh(f"{self.python} -m pip --version", check=True)

    def task_sandbox(self):
        """Task sandboxes run on this notebook's Python 3.13 (the scorer's sandbox is python:3.13-slim) with the
        scorer site built by gemma4-devset-build; pip is copied into its own dir (no ensurepip offline)."""
        import pip
        tgz = first(f"{INPUT}/**/scorer_site.tgz")
        assert tgz, "attach the output of kernel gemma4-devset-build (scorer_site.tgz)"
        SCORER_SITE.mkdir(parents=True, exist_ok=True)
        log.sh(f"tar -xzf {tgz} -C {SCORER_SITE}", timeout=1800, check=True)
        SANDBOX_PIP.mkdir(parents=True, exist_ok=True)
        src = pathlib.Path(pip.__file__).parent
        log.sh(f"cp -r {src} {SANDBOX_PIP}/ && cp -r {src.parent}/pip-*.dist-info {SANDBOX_PIP}/", check=True)
        py = os.path.realpath(sys.executable)
        env = {"python": py, "site": str(SCORER_SITE), "pip": str(SANDBOX_PIP)}
        pins = first(f"{INPUT}/**/overlays/pins.json")  # optional: kernel output gemma4-test-overlays
        if pins:
            root = pathlib.Path(pins).parent
            env.update(extras=str(root / "extras"), pins_root=str(root / "pins"), pins=json.loads(pathlib.Path(pins).read_text()))
            log(f"overlays: extras + pins for {len(env['pins'])} tasks from {root}")
        SANDBOX_ENV.write_text(json.dumps(env))
        sp = f"/tmp/sbx_check/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages"
        log.sh(f"{py} -m venv --without-pip --clear /tmp/sbx_check && printf '%s\\n%s\\n' {SCORER_SITE} {SANDBOX_PIP} "
               f"> {sp}/_s.pth && /tmp/sbx_check/bin/python3 -c 'import sys, pytest, pip; print(\"task sandbox\", "
               "sys.version.split()[0], \"pytest\", pytest.__version__, \"pip\", pip.__version__)'", check=True)

    def verify(self):
        log.sh(f"{self.python} -c 'import sys, torch, vllm, swegemma, adk_submission, google.adk, "
               "importlib.metadata as m; print(sys.version.split()[0], \"torch\", torch.__version__, \"vllm\", "
               "vllm.__version__, \"swegemma\", m.version(\"swegemma\"), \"adk-submission\", "
               "adk_submission.__version__, \"adk\", google.adk.__version__)'", check=True)
        # import needs pytest's pluggy, which the sandbox venv installs per task; here only check presence
        log.sh(f"{self.python} -c 'import importlib.util as u; "
               "print(\"test deps\", {m: bool(u.find_spec(m)) for m in (\"inline_snapshot\", \"dirty_equals\")})'")

    def prepare_training(self, gpus=True):
        """Scorer venv + the training wheels (peft 0.21.2, bitsandbytes 0.50.2: kernel output gemma4-train-wheels)."""
        t = time.time()
        if gpus:
            self.check_gpus()
        self.restore()
        train = [pathlib.Path(p) for p in sorted(map(str, INPUT.rglob("train_wheels/*.whl")))]
        assert train, "attach the output of kernel gemma4-train-wheels (kernel_sources)"
        self.install(self.stale_wheels() + train)
        self.ensure_pip()
        log.sh(f"{self.python} -c 'import torch, transformers, peft, bitsandbytes; print(\"torch\", torch.__version__, "
               "\"transformers\", transformers.__version__, \"peft\", peft.__version__, \"bnb\", bitsandbytes.__version__)'",
               check=True)
        log(f"training venv ready in {(time.time() - t) / 60:.1f} min")

    def prepare(self, gpus=True):
        t = time.time()
        if gpus:
            self.check_gpus()
        self.restore()
        self.install(self.stale_wheels() + self.test_dep_wheels())
        self.ensure_pip()
        self.task_sandbox()
        self.verify()
        log(f"scorer venv ready in {(time.time() - t) / 60:.1f} min")

## 4. Long-context fixes
Each fix is its own module and cell. `attn`, `mlp_tile`, `norm_ckpt` and `no_kv_store` can be switched at runtime through `config.LC`; `fix_head` and `fix_lora` are always on.

### `trainkit/fix_attention.py`
```
Fix: attention without dense T x T masks (exact). Replaces the "sdpa" entry HF's Gemma-4 attention looks up.
  sliding layers (50, window W=1024): queries in blocks of W attend to keys [block start - W + 1, block end) with a
      small (W x 2W) mask -- the same rule as HF's sliding mask (q - W < k <= q). HF instead builds a dense bool T x T
      mask (1 GiB at 32k), keeps it in the checkpoint kwargs, turns it into a 2T^2 bf16 bias inside SDPA, and computes
      all T x T scores: ~8x the attention FLOPs at 32k.
  global layers (10, 4 KV heads x 512): head_dim 512 > 256 makes HF skip SDPA's GQA and repeat K/V to all 32 heads
      (4 GB of K/V copies at 32k, kept for backward). Here: one causal SDPA call per KV group (8 query heads).
  Every block/group is its own checkpoint, so backward keeps only views of q/k/v, never the expanded copies.
Masks are disabled by passing NO_MASKS (a dict) as attention_mask: HF then skips mask creation for every layer.
Anything else (a real mask, a KV cache with q_len != k_len, decoding) goes to HF's own sdpa_attention_forward.
Also: the last full and sliding layer keep their full-length K/V in shared_kv_states for KV sharing; Gemma-4-31B
has num_kv_shared_layers=0, so nothing reads them (0.75 GiB at 32k, held by the checkpoint kwargs) -> turned off.
```

In [ ]:
%%writefile /kaggle/working/trainkit/fix_attention.py
"""
Fix: attention without dense T x T masks (exact). Replaces the "sdpa" entry HF's Gemma-4 attention looks up.
  sliding layers (50, window W=1024): queries in blocks of W attend to keys [block start - W + 1, block end) with a
      small (W x 2W) mask -- the same rule as HF's sliding mask (q - W < k <= q). HF instead builds a dense bool T x T
      mask (1 GiB at 32k), keeps it in the checkpoint kwargs, turns it into a 2T^2 bf16 bias inside SDPA, and computes
      all T x T scores: ~8x the attention FLOPs at 32k.
  global layers (10, 4 KV heads x 512): head_dim 512 > 256 makes HF skip SDPA's GQA and repeat K/V to all 32 heads
      (4 GB of K/V copies at 32k, kept for backward). Here: one causal SDPA call per KV group (8 query heads).
  Every block/group is its own checkpoint, so backward keeps only views of q/k/v, never the expanded copies.
Masks are disabled by passing NO_MASKS (a dict) as attention_mask: HF then skips mask creation for every layer.
Anything else (a real mask, a KV cache with q_len != k_len, decoding) goes to HF's own sdpa_attention_forward.
Also: the last full and sliding layer keep their full-length K/V in shared_kv_states for KV sharing; Gemma-4-31B
has num_kv_shared_layers=0, so nothing reads them (0.75 GiB at 32k, held by the checkpoint kwargs) -> turned off.
"""
import torch
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

from .config import LC

NO_MASKS = {"full_attention": None, "sliding_attention": None}
MASK = "bias"  # "bias": additive mask in q's dtype, row stride padded to 16 (L4 v2: the bool-mask variant's gradient
               # had cosine 0.55 vs HF's dense-mask path at 4k); "bool": the v2 variant, kept for the probe's unit test
_HF_SDPA = None
BLOCK = None  # query block for sliding layers (None = the window); the probe varies it as a rounding-only control
CAPTURE = None  # probe only: a list -> every call appends its arguments (lcprobe._real_width_diag)
NOISE = None  # probe only: (rel, seed) -> every attention output (HF's or ours) is multiplied by 1 + rel * U(-1, 1),
              # fixed per (seed, layer) so a checkpoint's recompute sees the same noise (rounding-sensitivity control)


def _grouped_causal(q, k, v, scale):
    """q [1, g, T, D]; k, v [1, 1, T, D] -> causal attention of g query heads over one KV head."""
    g = q.shape[1]
    return F.scaled_dot_product_attention(q, k.expand(-1, g, -1, -1).contiguous(), v.expand(-1, g, -1, -1).contiguous(),
                                          is_causal=True, scale=scale)


def _window_block(query, key, value, s, e, window, scale):
    """Queries [s, e) over keys [s - W + 1, e) with HF's rule q - W < k <= q. Slicing and the GQA expansion happen in
    here, so the checkpoint keeps only the full q/k/v it was given, not a per-block copy."""
    ks, g = max(0, s - window + 1), query.shape[1] // key.shape[1]
    q = query[:, :, s:e]
    k = key[:, :, ks:e].repeat_interleave(g, dim=1)  # same head order as HF's repeat_kv
    v = value[:, :, ks:e].repeat_interleave(g, dim=1)
    i = torch.arange(s, e, device=q.device)[:, None]
    j = torch.arange(ks, e, device=q.device)[None, :]
    mask = (j <= i) & (j > i - window)
    if MASK == "bool":
        return F.scaled_dot_product_attention(q, k, v, attn_mask=mask[None, None], scale=scale)
    L = e - ks  # the memory-efficient kernel wants the mask's row stride aligned to 16: pad, then slice
    bias = torch.full((e - s, -(-L // 16) * 16), float("-inf"), dtype=q.dtype, device=q.device)[:, :L]
    bias.masked_fill_(mask, 0.0)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=bias[None, None], scale=scale)


def _ckpt(fn, *args):
    return checkpoint(fn, *args, use_reentrant=False) if torch.is_grad_enabled() else fn(*args)


def _noisy(out, layer):
    rel, seed = NOISE
    g = torch.Generator(device=out.device).manual_seed(seed * 1000 + (layer or 0))
    u = torch.rand(out.shape, generator=g, device=out.device, dtype=torch.float32) * 2 - 1
    return (out.float() * (1 + rel * u)).to(out.dtype)


def lc_attention(module, query, key, value, attention_mask, dropout=0.0, scaling=None, is_causal=None, **kwargs):
    out, w = _attention(module, query, key, value, attention_mask, dropout, scaling, is_causal, **kwargs)
    return (out if NOISE is None else _noisy(out, getattr(module, "layer_idx", None))), w


def _attention(module, query, key, value, attention_mask, dropout, scaling, is_causal, **kwargs):
    T, Tk = query.shape[2], key.shape[2]
    if CAPTURE is not None:
        CAPTURE.append({"layer": getattr(module, "layer_idx", None), "T": T, "Tk": Tk, "q": query, "k": key, "v": value,
                        "mask": attention_mask, "dropout": dropout, "scaling": scaling, "is_causal": is_causal,
                        "kwargs": {k: (v if isinstance(v, (int, float, str, bool, type(None))) else type(v).__name__)
                                   for k, v in kwargs.items()}})
    if (not LC["attn"] or attention_mask is not None or T != Tk or T == 1 or dropout
            or not getattr(module, "is_causal", True)):
        return _HF_SDPA(module, query, key, value, attention_mask, dropout=dropout, scaling=scaling, is_causal=is_causal,
                        **kwargs)
    window = kwargs.get("sliding_window") or getattr(module, "sliding_window", None)
    hkv, g = key.shape[1], query.shape[1] // key.shape[1]
    if window and T > window:
        step = BLOCK or window
        out = torch.cat([_ckpt(_window_block, query, key, value, s, min(s + step, T), window, scaling)
                         for s in range(0, T, step)], dim=2)
    else:
        out = torch.cat([_ckpt(_grouped_causal, query[:, h * g:(h + 1) * g], key[:, h:h + 1], value[:, h:h + 1], scaling)
                         for h in range(hkv)], dim=1)
    return out.transpose(1, 2).contiguous(), None


def install(model):
    """Register lc_attention under "sdpa" (once) and set the K/V-store switch on every text attention module."""
    global _HF_SDPA
    from transformers.integrations.sdpa_attention import sdpa_attention_forward
    from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
    if _HF_SDPA is None:
        _HF_SDPA = sdpa_attention_forward
        ALL_ATTENTION_FUNCTIONS["sdpa"] = lc_attention
    for mod in model.modules():
        if type(mod).__name__ == "Gemma4TextAttention":
            if not hasattr(mod, "_hf_store_kv"):
                mod._hf_store_kv = mod.store_full_length_kv
            mod.store_full_length_kv = mod._hf_store_kv and not LC["no_kv_store"]

### `trainkit/fix_mlp.py`
```
Fix: tiled MLP (exact). Gemma-4's MLP is per position, so running it over sequence tiles gives the same outputs.
During a layer's backward recompute, HF keeps gate, up, act(gate) and their product for the whole sequence:
4 x T x 21504 bf16 = 5.6 GB at 32k, plus as much again in backward temporaries. With each tile its own checkpoint,
only the tile being differentiated is materialised (tile 4096: ~0.7 GB). Cost: one extra MLP forward per layer.
LoRA gradients from the tiles add up in .grad exactly as one big matmul would (up to summation order).
```

In [ ]:
%%writefile /kaggle/working/trainkit/fix_mlp.py
"""
Fix: tiled MLP (exact). Gemma-4's MLP is per position, so running it over sequence tiles gives the same outputs.
During a layer's backward recompute, HF keeps gate, up, act(gate) and their product for the whole sequence:
4 x T x 21504 bf16 = 5.6 GB at 32k, plus as much again in backward temporaries. With each tile its own checkpoint,
only the tile being differentiated is materialised (tile 4096: ~0.7 GB). Cost: one extra MLP forward per layer.
LoRA gradients from the tiles add up in .grad exactly as one big matmul would (up to summation order).
"""
import torch
from torch.utils.checkpoint import checkpoint

from .config import LC

TILE = 4096  # set from TrainSettings.mlp_tile by install()
_HF_FORWARD = None


def tiled_forward(self, x):
    T = x.shape[-2]
    if not (LC["mlp_tile"] and torch.is_grad_enabled() and T > TILE):
        return _HF_FORWARD(self, x)
    return torch.cat([checkpoint(_HF_FORWARD, self, x[..., a:a + TILE, :], use_reentrant=False)
                      for a in range(0, T, TILE)], dim=-2)


def install(tile):
    global _HF_FORWARD, TILE
    from transformers.models.gemma4.modeling_gemma4 import Gemma4TextMLP
    TILE = tile
    if _HF_FORWARD is None:
        _HF_FORWARD = Gemma4TextMLP.forward
        Gemma4TextMLP.forward = tiled_forward

### `trainkit/fix_norm.py`
```
Fix: checkpointed RMSNorm (exact: the same function, recomputed). Gemma4RMSNorm upcasts to fp32, so autograd keeps
fp32 copies of its input and intermediates: ~0.7 GB per hidden-size norm at 32k (4 per layer) plus the q/k/v norms.
As its own checkpoint a norm keeps only its bf16 input. Applies only when the input needs grad (training).
```

In [ ]:
%%writefile /kaggle/working/trainkit/fix_norm.py
"""
Fix: checkpointed RMSNorm (exact: the same function, recomputed). Gemma4RMSNorm upcasts to fp32, so autograd keeps
fp32 copies of its input and intermediates: ~0.7 GB per hidden-size norm at 32k (4 per layer) plus the q/k/v norms.
As its own checkpoint a norm keeps only its bf16 input. Applies only when the input needs grad (training).
"""
import torch
from torch.utils.checkpoint import checkpoint

from .config import LC

_HF_FORWARD = None


def ckpt_forward(self, hidden_states):
    if LC["norm_ckpt"] and torch.is_grad_enabled() and hidden_states.requires_grad:
        return checkpoint(_HF_FORWARD, self, hidden_states, use_reentrant=False)
    return _HF_FORWARD(self, hidden_states)


def install():
    global _HF_FORWARD
    from transformers.models.gemma4.modeling_gemma4 import Gemma4RMSNorm
    if _HF_FORWARD is None:
        _HF_FORWARD = Gemma4RMSNorm.forward
        Gemma4RMSNorm.forward = ckpt_forward

### `trainkit/fix_head.py`
```
Fix: the tied embedding / lm_head (262144 x 5376 bf16 = 2.82 GB) lives in pinned host RAM, outside FSDP (exact).
Before: it sat in the FSDP root unit, which FULL_SHARD never reshards after forward, so every rank held the whole
2.62 GiB through the step (plus its 0.7 GB shard). Now:
  embedding  rows are looked up on the CPU and copied to the GPU (T x 5376: 0.3 GB at 28k); frozen, no grad.
  loss       StreamCE: weighted next-token CE with Gemma's final-logit softcap, the vocabulary streamed through the
             GPU in tiles, two passes (1: running logsumexp + target logit, 2: dL/dh). Exact up to summation order;
             dL/dh is accumulated in fp32. ~5.6 GB of host->GPU copies per sequence (~0.5 s on PCIe 4).
Loss normalisation is the caller's: loss = sum_t w_t * ce_t / norm (norm = the optimizer step's total weight, so
every trained token counts the same however the step's sequences are split across ranks and micro-steps).
```

In [ ]:
%%writefile /kaggle/working/trainkit/fix_head.py
"""
Fix: the tied embedding / lm_head (262144 x 5376 bf16 = 2.82 GB) lives in pinned host RAM, outside FSDP (exact).
Before: it sat in the FSDP root unit, which FULL_SHARD never reshards after forward, so every rank held the whole
2.62 GiB through the step (plus its 0.7 GB shard). Now:
  embedding  rows are looked up on the CPU and copied to the GPU (T x 5376: 0.3 GB at 28k); frozen, no grad.
  loss       StreamCE: weighted next-token CE with Gemma's final-logit softcap, the vocabulary streamed through the
             GPU in tiles, two passes (1: running logsumexp + target logit, 2: dL/dh). Exact up to summation order;
             dL/dh is accumulated in fp32. ~5.6 GB of host->GPU copies per sequence (~0.5 s on PCIe 4).
Loss normalisation is the caller's: loss = sum_t w_t * ce_t / norm (norm = the optimizer step's total weight, so
every trained token counts the same however the step's sequences are split across ranks and micro-steps).
"""
import torch
import torch.nn as nn

V_TILE = 16384  # vocab rows per upload (176 MB bf16)
P_CHUNK = 4096  # positions per matmul (fp32 logits tile 4096 x 16384 = 268 MB)


class HostEmbedding(nn.Module):
    """Gemma4TextScaledWordEmbedding with its weight in (pinned) host RAM as a plain attribute, so FSDP never sees it."""

    def __init__(self, emb, device):
        super().__init__()
        w = emb.weight.detach().to("cpu")
        self.w = w.pin_memory() if device.type == "cuda" else w
        self.scale = emb.embed_scale.detach().to("cpu", w.dtype)  # HF: lookup * embed_scale in the weight dtype (L4 v1: was left on the GPU)
        self.device = device

    def forward(self, input_ids):
        rows = nn.functional.embedding(input_ids.cpu(), self.w) * self.scale
        return rows.to(self.device, non_blocking=True)


def move_to_host(m, device):
    """Swap the text embedding for HostEmbedding and drop the tied lm_head. Returns the host weight (= lm_head)."""
    base = m.base_model.model  # PeftModel -> LoraModel -> Gemma4ForConditionalGeneration
    lm = base.model.language_model
    head = base.get_output_embeddings()
    assert head.weight.data_ptr() == lm.embed_tokens.weight.data_ptr(), "expected tied embedding / lm_head"
    lm.embed_tokens = HostEmbedding(lm.embed_tokens, device)
    base.lm_head = nn.Identity()
    del head
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return lm.embed_tokens.w


def _tiles(w, dev):
    for v0 in range(0, w.shape[0], V_TILE):
        yield v0, w[v0:v0 + V_TILE].to(dev, non_blocking=True)


class StreamCE(torch.autograd.Function):
    """h [T, H] (GPU), w [V, H] (host), y [T] targets, wt [T] float weights (0 = not trained), cap softcap, norm."""

    @staticmethod
    def forward(ctx, h, w, y, wt, cap, norm):
        idx = (wt > 0).nonzero().squeeze(1)
        hs, ys, ws = h[idx], y[idx], wt[idx].float()
        n, dev = idx.numel(), h.device
        m = torch.full((n,), -float("inf"), device=dev)
        s = torch.zeros(n, device=dev)
        zt = torch.zeros(n, device=dev)

        def logits(a, wv):
            z = (hs[a:a + P_CHUNK] @ wv.T).float()
            return torch.tanh(z / cap) * cap if cap else z

        for v0, wv in _tiles(w, dev):  # pass 1: logsumexp and target logits
            for a in range(0, n, P_CHUNK):
                z = logits(a, wv)
                mz = torch.maximum(m[a:a + P_CHUNK], z.max(1).values)
                s[a:a + P_CHUNK] = s[a:a + P_CHUNK] * torch.exp(m[a:a + P_CHUNK] - mz) + torch.exp(z - mz[:, None]).sum(1)
                m[a:a + P_CHUNK] = mz
                yy = ys[a:a + P_CHUNK] - v0
                hit = (yy >= 0) & (yy < wv.shape[0])
                zt[a:a + P_CHUNK][hit] = z[hit.nonzero().squeeze(1), yy[hit]]
        lse = m + torch.log(s)
        loss = (ws * (lse - zt)).sum() / norm
        gs = torch.zeros(n, h.shape[1], device=dev, dtype=torch.float32)
        coef = ws / norm
        for v0, wv in _tiles(w, dev):  # pass 2: dL/dh = sum_v coef * (p_v - [v = y]) * dz_v/dl_v * w_v
            for a in range(0, n, P_CHUNK):
                z = logits(a, wv)
                g = torch.exp(z - lse[a:a + P_CHUNK, None])
                yy = ys[a:a + P_CHUNK] - v0
                hit = (yy >= 0) & (yy < wv.shape[0])
                g[hit.nonzero().squeeze(1), yy[hit]] -= 1.0
                if cap:
                    g *= 1.0 - (z / cap) ** 2  # d(cap * tanh(l / cap))/dl
                g *= coef[a:a + P_CHUNK, None]
                gs[a:a + P_CHUNK] += (g.to(h.dtype) @ wv).float()  # bf16 matmul as autograd would; fp32 sum over tiles
        gh = torch.zeros_like(h)
        gh[idx] = gs.to(h.dtype)
        ctx.save_for_backward(gh)
        return loss

    @staticmethod
    def backward(ctx, go):
        gh, = ctx.saved_tensors
        return gh * go, None, None, None, None, None


def reference_ce(h, w, y, wt, cap, norm):
    """Plain autograd over full logits (tests only: T x 262144 fp32)."""
    z = (h @ w.T).float()
    if cap:
        z = torch.tanh(z / cap) * cap
    ce = nn.functional.cross_entropy(z, y, reduction="none")
    return (wt.float() * ce).sum() / norm

### `trainkit/fix_lora.py`
```
Fix: LoRA weights outside FSDP, fp32 master weights, exact global loss normalisation.
Before: each layer's FSDP flat parameter mixed frozen NF4 storage with LoRA, so FSDP allocated gradients for the whole
flat parameter (~3.4 GiB/rank inferred), AdamW updated bf16 LoRA weights (updates below 2^-8 relative were lost), and
FSDP averaged per-rank mean losses (a 4k episode's tokens weighed ~7x a 28k episode's).
Now the LoRA tensors (~0.3 GB bf16) are FSDP-ignored and replicated: every rank starts from rank 0's init, backward
accumulates bf16 grads locally, and ShardedAdamW reduce-scatters them in fp32 (SUM: the loss is already divided by
the step's total weight), clips by the global norm, updates its 1/world fp32 master shard and all-gathers the result.
```

In [ ]:
%%writefile /kaggle/working/trainkit/fix_lora.py
"""
Fix: LoRA weights outside FSDP, fp32 master weights, exact global loss normalisation.
Before: each layer's FSDP flat parameter mixed frozen NF4 storage with LoRA, so FSDP allocated gradients for the whole
flat parameter (~3.4 GiB/rank inferred), AdamW updated bf16 LoRA weights (updates below 2^-8 relative were lost), and
FSDP averaged per-rank mean losses (a 4k episode's tokens weighed ~7x a 28k episode's).
Now the LoRA tensors (~0.3 GB bf16) are FSDP-ignored and replicated: every rank starts from rank 0's init, backward
accumulates bf16 grads locally, and ShardedAdamW reduce-scatters them in fp32 (SUM: the loss is already divided by
the step's total weight), clips by the global norm, updates its 1/world fp32 master shard and all-gathers the result.
"""
import torch
import torch.distributed as dist


def lora_params(m):
    return [p for n, p in m.named_parameters() if "lora_" in n]


def broadcast_init(params):
    for p in params:
        dist.broadcast(p.data, 0)


def _gloo():
    return dist.get_backend() == "gloo"


class ShardedAdamW:
    def __init__(self, params, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        self.params = params
        self.rank, self.world = dist.get_rank(), dist.get_world_size()
        self.numel = sum(p.numel() for p in params)
        self.n = -(-self.numel // self.world)  # shard size (padded)
        dev = params[0].device
        flat = self._flat([p.data for p in params], dev)
        self.master = torch.nn.Parameter(flat[self.rank * self.n:(self.rank + 1) * self.n].clone())
        self.opt = torch.optim.AdamW([self.master], lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        self.param_groups = self.opt.param_groups

    def _flat(self, tensors, dev):
        out = torch.zeros(self.n * self.world, device=dev, dtype=torch.float32)
        a = 0
        for t in tensors:
            out[a:a + t.numel()] = t.reshape(-1).float()
            a += t.numel()
        return out

    def step(self, max_norm=1.0):
        """-> global grad norm before clipping. Missing grads count as zero."""
        dev = self.master.device
        g = self._flat([p.grad if p.grad is not None else torch.zeros_like(p) for p in self.params], dev)
        shard = torch.empty(self.n, device=dev, dtype=torch.float32)
        if _gloo():  # gloo has no reduce_scatter
            dist.all_reduce(g)
            shard.copy_(g[self.rank * self.n:(self.rank + 1) * self.n])
        else:
            dist.reduce_scatter_tensor(shard, g)
        del g
        sq = (shard * shard).sum()
        dist.all_reduce(sq)
        norm = sq.sqrt()
        shard.mul_(torch.clamp(max_norm / (norm + 1e-6), max=1.0))
        self.master.grad = shard
        self.opt.step()
        self.master.grad = None
        full = torch.empty(self.n * self.world, device=dev, dtype=self.params[0].dtype)
        if _gloo():
            parts = [torch.empty(self.n, device=dev, dtype=full.dtype) for _ in range(self.world)]
            dist.all_gather(parts, self.master.data.to(full.dtype))
            full = torch.cat(parts)
        else:
            dist.all_gather_into_tensor(full, self.master.data.to(full.dtype))
        a = 0
        for p in self.params:
            p.data.copy_(full[a:a + p.numel()].view_as(p))
            a += p.numel()
        return norm

    def zero_grad(self):
        for p in self.params:
            p.grad = None

## 5. trainkit modules
One cell per module.

### `trainkit/__init__.py`
trainkit: LoRA SFT of gemma-4-31b-it on Kaggle 4x L4 with the recipe measured by the training probe (v4, Oct 10):

In [ ]:
%%writefile /kaggle/working/trainkit/__init__.py
"""
trainkit: LoRA SFT of gemma-4-31b-it on Kaggle 4x L4 with the recipe measured by the training probe (v4, Oct 10):
NF4 from the bf16 QAT weights, LoRA r16 on the 410 language-model linears, FSDP FULL_SHARD over 4 ranks,
memory-efficient SDPA (head_dim 512), activation offload for long windows, exact long-context fixes (fix_*.py).

Modules, in the order the notebook uses them:
  config    TrainSettings (one JSON shared by every process) and fixed paths
  data      TrainData: whole episodes (rs2.npz, float weights) or windows (train_*.npz), length-bucketed step plan
  fix_*     long-context fixes, one module each: attention without dense masks, tiled MLP, checkpointed RMSNorm,
            host-RAM embedding/lm_head + streamed CE, LoRA outside FSDP + ShardedAdamW (fp32 masters)
  model     load NF4 + LoRA + FSDP with the fixes installed; LossWrap (weighted, softcapped, streamed loss)
  lcprobe   LC1 probe: exactness of every fix and memory at 8-32k
  export    per-layer gather of the LoRA weights -> PEFT adapter that vLLM serves
  worker    the torchrun entry point: train loop, validation loss, periodic + final export
  launcher  runs torchrun inside the scorer venv, keeps per-rank logs
  main      orchestration after the env step (runs inside the venv)
smoke=True runs the same code on CPU (gloo, 2 ranks) with a tiny random Gemma-4 built from the real config.
Source of truth: kaggle/train/trainkit/; make_train_notebook.py turns it into a notebook.
"""

### `trainkit/config.py`
Training settings and fixed paths. Stdlib only.

In [ ]:
%%writefile /kaggle/working/trainkit/config.py
"""Training settings and fixed paths. Stdlib only."""
import json
import pathlib
import time
from dataclasses import asdict, dataclass, field

WORK = pathlib.Path("/kaggle/working")
INPUT = pathlib.Path("/kaggle/input")
CFG_FILE = WORK / "train_cfg.json"
DATA_FILE = WORK / "train_data.npz"       # written by the notebook's inputs cell (or found under /kaggle/input)
OUT_DIR = WORK / "train_out"              # adapters, metrics, per-rank logs
SMOKE_CONFIG = WORK / "g4cfg.json"        # real Gemma-4 config, for the tiny smoke model
TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
# Long-context fixes that can be switched at runtime (fix_attention / fix_mlp / fix_norm read these on every call;
# the probe toggles them to compare gradients). The structural fixes (fix_head, fix_lora) are always on.
LC = {"attn": True,      # fix_attention: block-local sliding attention + per-KV-group global attention, no dense masks
      "mlp_tile": True,  # fix_mlp: MLP over sequence tiles, each tile its own checkpoint
      "norm_ckpt": True,  # fix_norm: RMSNorm checkpointed (saves its bf16 input, not fp32 copies)
      "no_kv_store": True}  # fix_attention: drop the unused full-length K/V the last layers keep (num_kv_shared_layers=0)


@dataclass
class TrainSettings:
    name: str = "conv_lora"           # adapter name (adapters/<name>/ in the submission)
    rank: int = 16
    alpha: int = 32
    lr: float = 1e-4
    warmup_steps: int = 5
    grad_accum: int = 2               # optimizer step every grad_accum windows per rank (batch = 4 ranks x accum)
    epochs: float = 1.0
    max_windows: int | None = None    # cap (pipeline smoke runs)
    offload_from: int = 4097          # windows longer than this keep checkpoint activations in host RAM
    max_len: int = 28672              # episodes longer than this are dropped (never head-truncated: the task prompt is there)
    mlp_tile: int = 4096              # fix_mlp tile (tokens)
    val_every: int = 0                # optimizer steps between validation passes (0 = only at the end)
    save_every: int = 0               # optimizer steps between adapter exports (0 = only at the end)
    budget_h: float = 10.5            # stop taking steps after this, then export (session cap 12 h)
    seed: int = 0
    corr: str = "keep"                # corrected episodes (rs4.npz): keep | exclude (T2a control)
    corr_share: float = 0.0           # > 0: scale corrected episodes to this share of the train weight (T2b: 0.15)
    smoke: bool = False               # CPU, gloo, 2 ranks, tiny random model
    prefix_probe: bool = False        # PA1: run trainkit.prefix.probe instead of training (no data needed)
    lc_probe: bool = False            # LC1: run trainkit.lcprobe.probe (long-context fixes: exactness + memory)
    lc_tests: str = "UABC"            # which lcprobe tests run on L4 (D = rounding-only attention control)
    vablate: bool = False             # LP2: trainkit.vablate (val loss with/without the global layers' V delta)
    t0: float = field(default_factory=time.time)

    def hours_left(self):
        return self.budget_h - (time.time() - self.t0) / 3600

    def save(self, path=CFG_FILE):
        path.write_text(json.dumps(asdict(self), indent=1))

    @classmethod
    def load(cls, path=CFG_FILE):
        return cls(**json.loads(path.read_text()))

### `trainkit/data.py`
TrainData: whole episodes (rs2.npz: per-token float weights) or tokenized windows (train_*.npz: 0/1 mask) ->

In [ ]:
%%writefile /kaggle/working/trainkit/data.py
"""TrainData: whole episodes (rs2.npz: per-token float weights) or tokenized windows (train_*.npz: 0/1 mask) ->
length-bucketed steps of (input_ids, labels, weights) with each step's total weight known in advance."""
import random

import numpy as np
import torch

IGNORE = -100


class TrainData:
    def __init__(self, path, max_windows=None, vocab=None, seed=0, max_len=None, corr="keep", corr_share=0.0):
        d = np.load(path)
        self.ids = d["ids"]
        if "weight" in d:  # rs2.npz (kaggle/rftset/rs2_core.py): weight[t] = loss weight of token t, 0 = context
            self.w, self.starts, val = d["weight"], d["ep_starts"], d["val"].astype(bool)
        else:  # windows (posttrain/tokenize_windows.py): train[t] = 1 if token t is model-written
            self.w, self.starts, val = d["train"], d["starts"], d["val"].astype(bool)
        n = len(self.starts) - 1
        # rs4.npz (kaggle/rftset/rs4_core.py): corrected[i] = 1 for episodes built from failed runs (T2)
        self.corr = d["corrected"].astype(bool) if "corrected" in d else np.zeros(n, dtype=bool)
        self.scale = np.ones(n, dtype=np.float64)
        self.dropped_long = [i for i in range(n) if max_len and self.length(i) > max_len]
        drop = set(self.dropped_long)
        w = [i for i in range(n) if i not in drop and self.weight_sum(i) > 0]
        random.Random(seed).shuffle(w)
        self.train_w = [i for i in w if not val[i] and not (corr == "exclude" and self.corr[i])][:max_windows]
        self.val_w = [i for i in w if val[i]]
        self.val_sets = ({"onpolicy": [i for i in self.val_w if not self.corr[i]], "corrected": [i for i in self.val_w if self.corr[i]]}
                         if any(self.corr[i] for i in self.val_w) else {})
        wc = sum(self.weight_sum(i) for i in self.train_w if self.corr[i])
        wt = sum(self.weight_sum(i) for i in self.train_w)
        self.corr_share_raw = wc / wt if wt else 0.0
        if corr_share and wc and wt > wc:  # scale corrected episodes so they carry corr_share of the train weight
            self.scale[[i for i in self.train_w if self.corr[i]]] = corr_share * (wt - wc) / ((1 - corr_share) * wc)
        self.vocab = vocab  # smoke: fold ids into the tiny model's vocabulary
        self.seed = seed

    def length(self, w):
        return int(self.starts[w + 1] - self.starts[w])

    def weight_sum(self, w):
        """Total loss weight of window w's labels (tokens 1..T-1)."""
        return float(self.w[self.starts[w] + 1:self.starts[w + 1]].astype(np.float64).sum() * self.scale[w])

    def example(self, w, device):
        """-> (ids [1, T], labels [T-1], weights [T-1] float32): labels[t] predicts ids[t+1] with weight w[t+1]."""
        s, e = self.starts[w], self.starts[w + 1]
        ids = torch.from_numpy(self.ids[s:e].astype(np.int64))
        if self.vocab:
            ids = ids % self.vocab
        wts = torch.from_numpy(self.w[s + 1:e].astype(np.float32) * np.float32(self.scale[w]))
        lab = ids[1:].clone()
        lab[wts == 0] = IGNORE
        return ids.unsqueeze(0).to(device), lab.to(device), wts.to(device)

    def plan(self, windows, world, accum, epochs=1.0):
        """Every rank computes the same plan: a list of optimizer steps, each a list of `accum` micro-steps, each a list
        of `world` windows (one per rank). Per epoch: drop a random remainder, sort by length, cut into steps of
        world * accum consecutive windows (ranks of a step get similar lengths), shuffle the steps."""
        steps, per = [], world * accum
        n_ep = max(1, int(np.ceil(epochs)))
        for ep in range(n_ep):
            rng = random.Random(self.seed * 1000 + ep)
            ws = list(windows)
            rng.shuffle(ws)
            ws = sorted(ws[:len(ws) - len(ws) % per], key=self.length)
            ep_steps = [[ws[a + i * world:a + (i + 1) * world] for i in range(accum)] for a in range(0, len(ws), per)]
            rng.shuffle(ep_steps)
            if ep == n_ep - 1 and epochs < n_ep:
                ep_steps = ep_steps[:round(len(ep_steps) * (epochs - ep))]
            steps += ep_steps
        return steps

    def step_norm(self, step):
        """The step's total label weight over all ranks and micro-steps: the loss divisor that makes every trained
        token count the same."""
        return max(1e-6, sum(self.weight_sum(w) for micro in step for w in micro))

    def stats(self):
        lens = [self.length(w) for w in self.train_w]
        return {"train_windows": len(self.train_w), "val_windows": len(self.val_w), "dropped_long": len(self.dropped_long),
                "train_tokens": int(sum(lens)), "max_len": max(lens) if lens else 0,
                "train_weight": round(sum(self.weight_sum(w) for w in self.train_w), 1),
                "train_corrected": int(sum(self.corr[w] for w in self.train_w)), "val_corrected": len(self.val_sets.get("corrected", [])),
                "corr_share_raw": round(self.corr_share_raw, 4), "corr_scale": round(float(self.scale.max()), 3),
                "corr_share": round(sum(self.weight_sum(w) for w in self.train_w if self.corr[w]) /
                                    max(1e-6, sum(self.weight_sum(w) for w in self.train_w)), 4)}

### `trainkit/model.py`
Model: NF4 Gemma-4-31B + LoRA r16 + FSDP with the long-context fixes (fix_*.py), and the weighted streamed loss.

In [ ]:
%%writefile /kaggle/working/trainkit/model.py
"""Model: NF4 Gemma-4-31B + LoRA r16 + FSDP with the long-context fixes (fix_*.py), and the weighted streamed loss."""
import re
from functools import partial

import torch
import torch.nn as nn
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy
from torch.distributed.fsdp.wrap import lambda_auto_wrap_policy

from . import fix_attention, fix_head, fix_lora, fix_mlp, fix_norm
from .config import LC, TARGETS
from .data import IGNORE
from .fix_attention import NO_MASKS
from .fix_head import StreamCE

LORA_RE = r".*language_model.*\.(" + "|".join(TARGETS) + r")$"


def load_base(path, rank, smoke):
    """Every rank quantises the bf16 QAT weights to NF4 on its own GPU (~11 min); FSDP then shards them."""
    from transformers import AutoModelForImageTextToText
    if smoke:
        return AutoModelForImageTextToText.from_pretrained(path, dtype=torch.float32, attn_implementation="sdpa")
    from transformers import BitsAndBytesConfig
    q = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_storage=torch.bfloat16,  # FSDP can only shard 4-bit weights stored in the compute dtype
        llm_int8_skip_modules=["vision_tower", "embed_vision", "multi_modal_projector", "audio_tower", "embed_audio",
                               "lm_head"])
    return AutoModelForImageTextToText.from_pretrained(path, dtype=torch.bfloat16, attn_implementation="sdpa",
                                                       quantization_config=q, device_map={"": rank})


def add_lora(m, r, alpha, dtype):
    from peft import LoraConfig, get_peft_model
    m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    # no enable_input_require_grads: non-reentrant checkpointing does not need it, and the embedding is frozen
    m = get_peft_model(m, LoraConfig(r=r, lora_alpha=alpha, lora_dropout=0.0, bias="none", target_modules=LORA_RE,
                                     task_type="CAUSAL_LM"))
    for n, p in m.named_parameters():
        if "lora_" in n:
            p.data = p.data.to(dtype)  # FSDP flat params need one dtype per unit
    return m


class LossWrap(nn.Module):
    """FSDP root. The tied embedding / lm_head is in host RAM (fix_head), so the root holds only norms and buffers."""

    def __init__(self, m, cap, head):
        super().__init__()
        self.m, self.cap = m, cap
        self.head = [head]  # a list, so it is neither a parameter nor a buffer FSDP would move to the GPU

    def forward(self, ids, labels, wts=None, norm=None, prefix=0, hidden=False):
        """loss = sum_t wts_t * ce_t / norm over labels [T-1] (wts default: 1 where labels != IGNORE; norm default:
        the local weight sum, i.e. this sequence's mean). prefix > 0: ids[:, :prefix] only feed a frozen KV cache.
        hidden: return the final hidden states h[:-1] instead (probe)."""
        lm = self.m.base_model.model.model.language_model
        if wts is None:
            wts = (labels != IGNORE).float()
        if norm is None:
            norm = wts.sum().clamp(min=1.0)
        if prefix:
            from .prefix import suffix_hidden
            h = suffix_hidden(lm, ids, prefix)
        else:
            h = lm(input_ids=ids, attention_mask=NO_MASKS if LC["attn"] else None, use_cache=False).last_hidden_state[0]
        if hidden:
            return h[:-1]
        y = labels.clamp(min=0)
        return StreamCE.apply(h[:-1], self.head[0], y, wts, self.cap, float(norm))


def wrap_fsdp(m, smoke, mlp_tile=4096):
    """Install the long-context fixes, then one FSDP unit per decoder layer, the vision tower in its own (never
    gathered) unit, the rest in the root. LoRA tensors are FSDP-ignored (fix_lora); the head lives on the host."""
    dev = torch.device("cpu") if smoke else torch.device("cuda", torch.cuda.current_device())
    fix_attention.install(m)
    fix_mlp.install(mlp_tile)
    fix_norm.install()
    head = fix_head.move_to_host(m, dev)
    lora = fix_lora.lora_params(m)
    fix_lora.broadcast_init(lora)  # replicated tensors: every rank must start from the same init
    lm = next(mm for n, mm in m.named_modules() if n.endswith("language_model") and "vision" not in n)
    layer_cls = type(lm.layers[0])
    vision = [mm for n, mm in m.named_modules() if n.endswith("vision_tower") or n.endswith("embed_vision")]
    types = tuple({layer_cls} | {type(v) for v in vision})
    cap = getattr(m.config.get_text_config(), "final_logit_softcapping", None)
    fm = FSDP(LossWrap(m, cap, head), auto_wrap_policy=partial(lambda_auto_wrap_policy, lambda_fn=lambda x: isinstance(x, types)),
              sharding_strategy=ShardingStrategy.FULL_SHARD, use_orig_params=True,
              device_id=torch.device("cpu") if smoke else torch.cuda.current_device(), limit_all_gathers=True,
              sync_module_states=False, ignored_states=lora)
    return fm, layer_cls, lora


def lora_modules(m):
    return sum(1 for n, _ in m.named_parameters() if ".lora_A." in n and re.match(LORA_RE, n.split(".lora_A.")[0]))

### `trainkit/prefix.py`
Idea A (PA1 probe): prefix-cached truncated backprop, so late-run steps train in their real 20-30k context.

In [ ]:
%%writefile /kaggle/working/trainkit/prefix.py
"""
Idea A (PA1 probe): prefix-cached truncated backprop, so late-run steps train in their real 20-30k context.
The first P tokens run under no_grad (chunked prefill into the model's own HF cache), the cache is frozen (update()
returns cache + new keys without storing them, so checkpoint recompute in backward sees the same state), then
forward + backward runs on the last S tokens only. LoRA shapes the prefix forward; gradients stop at the cache.
Gemma-4-31B: 50 of 60 layers are sliding-window (1024), so the cache is ~0.8 GB for those + ~1-2 GB per 24k tokens
for the 10 global layers (4 KV heads x 512).
probe(): correctness (cache path loss == full-sequence loss on the same targets), a short overfit run, and peak
memory / time over a (P, S) grid on random tokens. Rank 0 writes OUT_DIR/prefix_probe.json after every test.
"""
import contextlib
import functools
import gc
import glob
import json
import time

import torch
import torch.nn as nn

from .config import OUT_DIR
from .data import IGNORE

PREFILL_CHUNK = 4096  # L4 v1: 8192-token no-grad chunks + a full-length cache OOMed at P=9216
TIMING = {}  # last prefill wall time (probe: prefill vs trained tokens/s)


def freeze_cache(cache):
    """Make every cache layer read-only: update() returns (stored + new) without storing, lengths stay at P."""
    for layer in cache.layers:
        k0, v0 = layer.keys.detach(), layer.values.detach()

        def update(key_states, value_states, cache_kwargs=None, k0=k0, v0=v0):
            return torch.cat([k0, key_states], dim=-2), torch.cat([v0, value_states], dim=-2)
        layer.update = update  # plain closure: a bound method would make layer <-> method a cycle that keeps the cache
    return cache


def windowed_cache(cfg):
    """DynamicCache with a window-bounded layer for every sliding layer. transformers 5.13's default (and
    DynamicCache(config=...)) gave plain DynamicLayers that keep the whole prefix: ~16 KB/token x 50 layers."""
    from transformers.cache_utils import DynamicCache, DynamicLayer, DynamicSlidingWindowLayer
    cache = DynamicCache()
    cache.layers = [DynamicSlidingWindowLayer(sliding_window=cfg.sliding_window) if t == "sliding_attention" else DynamicLayer()
                    for t in cfg.layer_types]
    if hasattr(cache, "layer_class_to_replicate"):
        cache.layer_class_to_replicate = None  # layers are fixed; never append
    return cache


def prefix_cache(lm, ids, chunk=PREFILL_CHUNK):
    """No-grad chunked prefill of ids [1, P] into the model's own cache (sliding layers keep only their window)."""
    t = time.perf_counter()
    was = lm.training
    lm.eval()  # HF turns use_cache off under gradient checkpointing in train mode
    cache = windowed_cache(lm.config)
    with torch.no_grad():
        for a in range(0, ids.shape[1], chunk):
            pos = torch.arange(a, min(a + chunk, ids.shape[1]), device=ids.device)
            cache = lm(input_ids=ids[:, a:a + chunk], past_key_values=cache, use_cache=True, cache_position=pos,
                       position_ids=pos.unsqueeze(0)).past_key_values
    lm.train(was)
    if ids.is_cuda:
        torch.cuda.synchronize()
    TIMING["prefill_s"] = time.perf_counter() - t
    TIMING["cache_gb"] = sum(x.numel() * x.element_size() for layer in cache.layers for x in (layer.keys, layer.values)) / 2 ** 30
    TIMING["cache_layers"] = {k: sum(type(l).__name__ == k for l in cache.layers) for k in {type(l).__name__ for l in cache.layers}}
    return freeze_cache(cache)


@contextlib.contextmanager
def checkpoint_keeps_cache():
    """HF's GradientCheckpointingLayer sets past_key_values=None in train mode (a cache that grows would be appended
    twice by the recompute). Ours is frozen, so recompute is safe: checkpoint the layer with its kwargs unchanged."""
    from transformers.modeling_layers import GradientCheckpointingLayer
    orig = GradientCheckpointingLayer.__call__

    def call(self, *args, **kwargs):
        if self.gradient_checkpointing and self.training:
            return self._gradient_checkpointing_func(functools.partial(nn.Module.__call__, self, **kwargs), *args)
        return nn.Module.__call__(self, *args, **kwargs)
    GradientCheckpointingLayer.__call__ = call
    try:
        yield
    finally:
        GradientCheckpointingLayer.__call__ = orig


def suffix_hidden(lm, ids, prefix):
    """Hidden states of ids[:, prefix:] attending to a frozen cache of ids[:, :prefix]."""
    cache = prefix_cache(lm, ids[:, :prefix])
    pos = torch.arange(prefix, ids.shape[1], device=ids.device)
    with checkpoint_keeps_cache():
        return lm(input_ids=ids[:, prefix:], past_key_values=cache, use_cache=False, cache_position=pos,
                  position_ids=pos.unsqueeze(0)).last_hidden_state[0]


# ------------------------------------------------------------------ probe
_REAL = {}


def _real_ids(T, seed):
    """First T tokens of a real RS2 episode (attached kernel output gemma4-rftset-rs2), else None. Random tokens give
    losses ~16-18 near the logit softcap, where bf16 noise looks like error (L4 v2)."""
    if "eps" not in _REAL:
        import numpy as np
        f = sorted(glob.glob("/kaggle/input/**/rs2.npz", recursive=True))
        _REAL["eps"] = []
        if f:
            d = np.load(f[0])
            st = d["ep_starts"]
            _REAL["eps"] = [d["ids"][st[k]:st[k + 1]] for k in range(len(st) - 1)]
    long = [e for e in _REAL["eps"] if len(e) >= T]
    return torch.from_numpy(long[seed % len(long)][:T].astype("int64")).unsqueeze(0) if long else None


def _example(T, P, vocab, device, seed, real=False):
    g = torch.Generator().manual_seed(seed)  # same tokens on every rank (FSDP needs equal shapes, not equal data)
    ids = _real_ids(T, seed) if real else None
    ids = torch.randint(10, vocab, (1, T), generator=g) if ids is None else ids
    full_lab = ids[0, 1:].clone()
    full_lab[:P] = IGNORE  # full sequence: train only targets after the prefix
    return ids.to(device), full_lab.to(device), ids[0, P + 1:].clone().to(device)


def probe(tr, smoke):
    fm, dev, log = tr.fm, tr.dev, tr.log
    vocab = min(tr.vocab, 50000)
    out = {"smoke": smoke}
    path = OUT_DIR / "prefix_probe.json"

    def save():
        if tr.rank == 0:
            path.write_text(json.dumps(out, indent=1))

    def peak():
        return 0 if smoke else round(torch.cuda.max_memory_allocated() / 2 ** 30, 2)

    def reset():
        gc.collect()
        if not smoke:
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

    # 1. correctness: same targets, full sequence vs prefix cache (crosses the sliding window when P > window)
    corr = []
    for P, S in ([(40, 24), (64, 48)] if smoke else [(3072, 2048), (9216, 2048)]):
        ids, full_lab, suf_lab = _example(P + S, P, vocab, dev, seed=P, real=not smoke)
        fm.eval()
        with torch.no_grad():
            lf = float(fm(ids, full_lab))
            lp = float(fm(ids, suf_lab, prefix=P))
        fm.train()  # train mode = the gradient-checkpointing path (v1 smoke crashed only here); no_grad: a forward without
        with torch.no_grad():  # backward would leave FSDP's root unsharded and its graph alive (L4 v1 OOM)
            lt = float(fm(ids, suf_lab, prefix=P))
        cache_info = {"cache_gb": round(TIMING["cache_gb"], 3), "cache_layers": TIMING["cache_layers"]}
        with torch.no_grad():  # negative control: same targets with no prefix at all must differ from full
            ls = float(fm(ids[:, P:], suf_lab))
        corr.append({"P": P, "S": S, "loss_full": round(lf, 5), "loss_prefix": round(lp, 5), "abs_diff": round(abs(lf - lp), 5),
                     "loss_prefix_train": round(lt, 5), "abs_diff_train": round(abs(lf - lt), 5),
                     "loss_suffix_only": round(ls, 5), "abs_diff_suffix_only": round(abs(lf - ls), 5), **cache_info,
                     "peak_gb": peak()})
        reset()
        log(event="prefix_correct", **corr[-1])
    out["correctness"] = corr
    out["correctness_tokens"] = "real (rs2.npz)" if not smoke and _REAL.get("eps") else "random"
    cfg = fm.module.m.config.get_text_config()
    out["config"] = {k: getattr(cfg, k, None) for k in ("attn_logit_softcapping", "final_logit_softcapping", "sliding_window",
                                                       "attention_k_eq_v")}
    if not smoke:
        out["config"]["sdpa"] = {"flash": torch.backends.cuda.flash_sdp_enabled(), "mem_efficient": torch.backends.cuda.mem_efficient_sdp_enabled(),
                                 "math": torch.backends.cuda.math_sdp_enabled()}
    save()

    reset()
    # 1b. gradient fidelity: LoRA grads of the truncated path vs full backprop through the prefix (same targets)
    P, S = (32, 32) if smoke else (4096, 4096)
    ids, full_lab, suf_lab = _example(P + S, P, vocab, dev, seed=11)
    grads = []
    for path_ in ("full", "prefix"):
        with torch.autograd.graph.save_on_cpu(pin_memory=not smoke):
            loss = fm(ids, full_lab) if path_ == "full" else fm(ids, suf_lab, prefix=P)
        loss.backward()
        grads.append([q.grad.detach().float().clone() if q.grad is not None else None
                      for q in fm.parameters() if q.requires_grad])
        for q in fm.parameters():
            q.grad = None
    acc = torch.zeros(3, device=dev, dtype=torch.float64)  # dot, |full|^2, |prefix|^2 over this rank's shards
    for a, b in zip(*grads):
        if a is not None and b is not None:
            acc += torch.stack([(a * b).sum(), (a * a).sum(), (b * b).sum()]).double()
    torch.distributed.all_reduce(acc)
    out["grad_fidelity"] = {"P": P, "S": S, "cosine": round(float(acc[0] / (acc[1].sqrt() * acc[2].sqrt() + 1e-30)), 4),
                            "norm_ratio_prefix_over_full": round(float(acc[2].sqrt() / (acc[1].sqrt() + 1e-30)), 4)}
    log(event="prefix_grad_fidelity", **out["grad_fidelity"])
    del grads
    save()

    # 2. overfit sanity: a few optimizer steps on one example through the prefix path; loss must fall
    P, S = (48, 32) if smoke else (16384, 4096)
    ids, _, suf_lab = _example(P + S, P, vocab, dev, seed=7)
    opt = torch.optim.AdamW([p for p in fm.parameters() if p.requires_grad], lr=1e-3 if smoke else 2e-4)
    losses = []
    reset()
    for _ in range(4):
        with torch.autograd.graph.save_on_cpu(pin_memory=not smoke):
            loss = fm(ids, suf_lab, prefix=P)
        loss.backward()
        fm.clip_grad_norm_(1.0)
        opt.step()
        opt.zero_grad(set_to_none=True)
        losses.append(round(float(loss), 4))
    out["overfit"] = {"P": P, "S": S, "losses": losses, "falls": losses[-1] < losses[0], "peak_gb": peak()}
    log(event="prefix_overfit", **out["overfit"])
    save()
    del opt

    # 3. memory / time grid, ascending total length; stop at the first OOM (later configs are larger)
    grid = [(0, 32), (32, 32), (64, 32)] if smoke else [(0, 8192), (16384, 4096), (16384, 8192), (24576, 4096), (24576, 6144),
                                                      (28672, 4096), (24576, 8192)]
    out["grid"] = []
    for P, S in grid:
        ids, full_lab, suf_lab = _example(P + S, P, vocab, dev, seed=P + S)
        reset()
        t = time.perf_counter()
        row = {"P": P, "S": S}
        try:
            with torch.autograd.graph.save_on_cpu(pin_memory=not smoke):
                loss = fm(ids, suf_lab, prefix=P) if P else fm(ids, full_lab)
            loss.backward()
            if not smoke:
                torch.cuda.synchronize()
            sec = time.perf_counter() - t
            pre = TIMING.get("prefill_s", 0.0) if P else 0.0
            row.update(ok=True, sec=round(sec, 1), prefill_s=round(pre, 1), peak_gb=peak(), loss=round(float(loss), 4),
                       cache_gb=round(TIMING.get("cache_gb", 0.0), 3) if P else 0.0,
                       prefill_tok_s=round(P / pre, 1) if P and pre else None,
                       trained_tok_s=round(S / max(1e-6, sec - pre), 1), host_pinned_gb=None if smoke else
                       round(torch.cuda.memory_stats().get("host_alloc_bytes.all.peak", 0) / 2 ** 30, 2))
        except torch.cuda.OutOfMemoryError as e:
            row.update(ok=False, error=f"OOM: {str(e)[:200]}", peak_gb=peak())
        for p in fm.parameters():
            p.grad = None
        flag = torch.tensor([0.0 if row["ok"] else 1.0], device=dev)
        torch.distributed.all_reduce(flag, op=torch.distributed.ReduceOp.MAX)
        row["any_rank_failed"] = bool(float(flag))
        out["grid"].append(row)
        log(event="prefix_grid", **row)
        save()
        if row["any_rank_failed"]:
            break
    # 4. unequal prefixes across ranks (same number of prefill chunks), as real batches will have; watched by NCCL timeout
    if not any(r["any_rank_failed"] for r in out["grid"]):
        P = (32 + 4 * tr.rank) if smoke else (17408 + 1024 * tr.rank)
        S = 32 if smoke else 4096
        ids, _, suf_lab = _example(P + S, P, vocab, dev, seed=99 + tr.rank)
        reset()
        t = time.perf_counter()
        with torch.autograd.graph.save_on_cpu(pin_memory=not smoke):
            loss = fm(ids, suf_lab, prefix=P)
        loss.backward()
        for q in fm.parameters():
            q.grad = None
        out["unequal_prefix"] = {"ok": True, "P_rank0": P, "S": S, "sec": round(time.perf_counter() - t, 1), "peak_gb": peak()}
        log(event="prefix_unequal", **out["unequal_prefix"])
    out["done"] = True
    save()

### `trainkit/lcprobe.py`
LC1 probe: are the long-context fixes exact, and does exact full backprop fit at 24-32k on 4x L4?

In [ ]:
%%writefile /kaggle/working/trainkit/lcprobe.py
"""
LC1 probe: are the long-context fixes exact, and does exact full backprop fit at 24-32k on 4x L4?
Smoke (CPU, fp32, tiny Gemma-4, window 16, MLP tile 32): the whole new stack (block attention, tiled MLP,
checkpointed norms, host head + StreamCE, replicated LoRA + ShardedAdamW, global normalisation) against plain HF
autograd on an unpatched copy of the model (HF masks, full logits, torch AdamW); each runtime fix switched off alone;
then a short end-to-end train run on a synthetic rs2-format file (plan, validation, export).
L4 (real RS2 tokens; Gemma-4-31B NF4):
  A  StreamCE vs full-logit autograd on 2k real hidden states (loss, dL/dh)
  B  LoRA gradients at T=4096 with each runtime fix alone and all together vs all off (run twice = noise floor)
  C  all fixes on, one fwd+bwd at T = 8k, 24k, 28k, 32k with a per-component memory breakdown at the peak (allocator
     trace replay), stop at the first OOM; then one ShardedAdamW step on the last gradients
Rank 0 writes OUT_DIR/lc_probe.json after every test.
"""
import gc
import glob
import json
import time
import zlib

import numpy as np
import torch
import torch.distributed as dist

from . import fix_attention, fix_head
from .config import LC, OUT_DIR
from .data import IGNORE, TrainData
from .fix_head import StreamCE, reference_ce
from .fix_lora import ShardedAdamW

FLAGS = tuple(LC)
_EPS = {}


def set_flags(fm, **on):
    for k in FLAGS:
        LC[k] = bool(on.get(k, False))
    fix_attention.install(fm)


def _key(n):
    return n.replace("_fsdp_wrapped_module.", "").removeprefix("m.").replace(".default.", ".")


def _randomize(named, std=0.05, only="lora_"):
    """Same deterministic values for the same LoRA tensor in any model and on every rank (PEFT's B = 0 init gives A
    no gradient and leaves the forward untouched by LoRA)."""
    for n, p in named:
        if only in n:
            g = torch.Generator().manual_seed(zlib.crc32(_key(n).encode()))
            p.data.copy_((torch.randn(p.shape, generator=g) * std).to(p.dtype))


# ------------------------------------------------------------------ inputs
def _episodes():
    if "eps" not in _EPS:
        f = sorted(glob.glob("/kaggle/input/**/rs2.npz", recursive=True))
        _EPS["eps"] = []
        if f:
            d = np.load(f[0])
            st, ids, w = d["ep_starts"], d["ids"], d["weight"]
            _EPS["eps"] = [(ids[st[k]:st[k + 1]], w[st[k]:st[k + 1]]) for k in range(len(st) - 1)]
            _EPS["eps"].sort(key=lambda e: -len(e[0]))
    return _EPS["eps"]


def _seq(T, k, dev, vocab=None, tail=False):
    """Sequence k of length T: a real RS2 episode (longest first; concatenated beyond the longest), else random."""
    eps = _episodes()
    if eps:
        long = [e for e in eps if len(e[0]) >= T]
        parts = [long[k % len(long)]] if long else [eps[(k + j) % len(eps)] for j in range(len(eps))]
        end = int(np.nonzero(parts[0][1])[0][-1]) + 1 if tail and long else T
        cut = slice(max(0, end - T), max(0, end - T) + T)
        ids = torch.from_numpy(np.concatenate([e[0] for e in parts])[cut].astype(np.int64))
        wts = torch.from_numpy(np.concatenate([e[1] for e in parts])[cut][1:].astype(np.float32))
        assert ids.numel() == T, f"RS2 holds fewer than {T} tokens"
    else:
        g = torch.Generator().manual_seed(1000 + k)
        ids = torch.randint(10, vocab, (T,), generator=g)
        wts = (torch.rand(T - 1, generator=g) < 0.6).float() * torch.rand(T - 1, generator=g) * 3
    lab = ids[1:].clone()
    lab[wts == 0] = IGNORE
    return ids[None].to(dev), lab.to(dev), wts.to(dev)


# ------------------------------------------------------------------ gradient comparison
def _grads(tr):
    return [p.grad.detach().float().clone() if p.grad is not None else torch.zeros_like(p, dtype=torch.float32)
            for p in tr.lora]


def _compare(a, b, reduce=True):
    """a vs reference b (lists of tensors): cosine, norm ratio, worst per-tensor relative error; summed over ranks
    (each rank holds its own sequence's grads) when reduce."""
    dev = a[0].device
    acc = torch.zeros(3, device=dev, dtype=torch.float64)
    worst = torch.zeros(1, device=dev, dtype=torch.float64)
    for x, y in zip(a, b):
        acc += torch.stack([(x * y).sum(), (x * x).sum(), (y * y).sum()]).double()
        if y.norm() > 0:
            worst = torch.maximum(worst, ((x - y).norm() / y.norm()).double().reshape(1))
    if reduce:
        dist.all_reduce(acc)
        dist.all_reduce(worst, op=dist.ReduceOp.MAX)
    return {"cosine": round(float(acc[0] / (acc[1].sqrt() * acc[2].sqrt() + 1e-30)), 6),
            "norm_ratio": round(float(acc[1].sqrt() / (acc[2].sqrt() + 1e-30)), 6), "worst_tensor_rel": round(float(worst), 6)}


HOST = {}


def _host_avail_gb():
    with open("/proc/meminfo") as f:
        return round(int(next(l for l in f if l.startswith("MemAvailable")).split()[1]) / 2 ** 20, 1)


def _fwd_bwd(tr, ids, lab, wts, norm, smoke):
    for p in tr.lora:
        p.grad = None
    with torch.autograd.graph.save_on_cpu(pin_memory=False):  # pageable: pinned blocks round up (host OOM at 28k, v2)
        loss = tr.fm(ids, lab, wts, norm)
    HOST["avail_after_fwd_gb"] = _host_avail_gb()  # all checkpoint inputs are on the host now (node-wide figure)
    loss.backward()
    return float(loss)


# ------------------------------------------------------------------ real-width diagnostic (CPU, no checkpoint)
def _real_width_diag(tr, T=2100):
    """L4 v2/v3: every attention path matched an fp32 reference in isolation, yet the model's loss/grads differ between
    attn off (HF masks) and on (NO_MASKS). A 3-layer Gemma-4 at the real width and config (2 sliding + 1 full layer,
    random init, fp32, CPU math kernel) run both ways: per-layer output difference, what each attention call receives,
    and HF's attention re-run vs lc_attention on the very tensors the model passed."""
    from transformers import AutoConfig, AutoModelForImageTextToText
    from .config import SMOKE_CONFIG
    real = json.loads(SMOKE_CONFIG.read_text())
    tc = real["text_config"]
    tc.update(num_hidden_layers=3, layer_types=["sliding_attention", "sliding_attention", "full_attention"],
              vocab_size=1024, vocab_size_per_layer_input=1024)
    real["vision_config"].update(hidden_size=32, intermediate_size=64, num_hidden_layers=1, num_attention_heads=2,
                                 num_key_value_heads=2, head_dim=16, global_head_dim=16)
    torch.manual_seed(0)
    cfg = AutoConfig.for_model(**{k: v for k, v in real.items() if k != "transformers_version"})
    m = AutoModelForImageTextToText.from_config(cfg, attn_implementation="sdpa").eval()
    fix_attention.install(m)
    lm = m.model.language_model
    ids = torch.randint(10, 1000, (1, T), generator=torch.Generator().manual_seed(5))
    res = {"T": T, "text_config": {k: tc.get(k) for k in ("hidden_size", "num_attention_heads", "num_key_value_heads",
                                                          "num_global_key_value_heads", "head_dim", "global_head_dim",
                                                          "sliding_window", "attention_k_eq_v", "use_bidirectional_attention")}}
    outs = {}
    hooks = [layer.register_forward_hook(lambda mod, a, o, i=i: outs.setdefault(mode, {}).__setitem__(i, o.detach().clone()))
             for i, layer in enumerate(lm.layers)]
    calls = {}
    with torch.no_grad():
        for mode in ("off", "on"):
            LC["attn"] = mode == "on"
            fix_attention.CAPTURE = []
            h = lm(input_ids=ids, attention_mask=fix_attention.NO_MASKS if mode == "on" else None, use_cache=False).last_hidden_state
            outs.setdefault(mode, {})["final"] = h.clone()
            calls[mode] = fix_attention.CAPTURE
            fix_attention.CAPTURE = None
    for hk in hooks:
        hk.remove()
    res["layer_rel_diff"] = {str(k): round(float((outs["on"][k] - outs["off"][k]).norm() / outs["off"][k].norm()), 8)
                             for k in outs["off"]}
    i, j = torch.arange(T)[:, None], torch.arange(T)[None, :]
    res["calls"] = []
    for c_off, c_on in zip(calls["off"], calls["on"]):
        row = {"layer": c_off["layer"], "T": c_off["T"], "Tk": c_off["Tk"], "q_shape": list(c_off["q"].shape),
               "k_shape": list(c_off["k"].shape), "q_contig": c_off["q"].is_contiguous(), "q_stride": list(c_off["q"].stride()),
               "dropout": c_off["dropout"], "scaling": c_off["scaling"], "is_causal": c_off["is_causal"],
               "kwargs": c_off["kwargs"], "mask_off": None if c_off["mask"] is None else
               {"dtype": str(c_off["mask"].dtype), "shape": list(c_off["mask"].shape)},
               "mask_on": None if c_on["mask"] is None else str(type(c_on["mask"])),
               "inputs_equal_off_on": all(torch.equal(c_off[x], c_on[x]) for x in "qkv")}
        mk = c_off["mask"]
        if mk is not None:
            W = c_off["kwargs"].get("sliding_window")
            allowed = mk[0, 0] if mk.dtype == torch.bool else mk[0, 0] == 0
            rule = (j <= i) & (j > i - W) if W else (j <= i)
            row["mask_disagree_vs_rule"] = int((allowed != rule).sum())
        mod = lm.layers[c_off["layer"]].self_attn
        LC["attn"] = False
        o_hf, _ = fix_attention._HF_SDPA(mod, c_off["q"], c_off["k"], c_off["v"], c_off["mask"], dropout=0.0,
                                         scaling=c_off["scaling"], **{k: v for k, v in c_off["kwargs"].items()
                                                                      if k == "sliding_window"})
        LC["attn"] = True
        o_lc, _ = fix_attention.lc_attention(mod, c_off["q"], c_off["k"], c_off["v"], None, scaling=c_off["scaling"],
                                             **{k: v for k, v in c_off["kwargs"].items() if k == "sliding_window"})
        row["attn_out_rel_diff_same_inputs"] = round(float((o_lc - o_hf).norm() / o_hf.norm()), 8)
        res["calls"].append(row)
    del m, outs, calls
    gc.collect()
    tr.log(event="lc_R", layer_rel_diff=res["layer_rel_diff"])
    return res


# ------------------------------------------------------------------ attention unit test (no model)
def _attn_unit(dev, smoke):
    """lc_attention (bias mask, bool mask) and HF's sdpa path with its dense mask, each vs an fp32 reference, at the
    real head shapes: forward error and q/k/v gradient cosine. Lengths include one that is not a multiple of 16."""
    from types import SimpleNamespace
    torch.manual_seed(0)
    cases = ([("sliding", 4, 2, 16, 16), ("global", 4, 1, 32, None)] if smoke else
             [("sliding", 32, 16, 256, 1024), ("global", 32, 4, 512, None)])
    rows, dt = [], torch.float32 if smoke else torch.bfloat16
    for kind, H, KV, D, W in cases:
        for T in ((100, 103) if smoke else (4096, 4100)):
            q, k, v = (torch.randn(1, n, T, D, device=dev) * D ** -0.25 for n in (H, KV, KV))
            mod = SimpleNamespace(is_causal=True, sliding_window=W, num_key_value_groups=H // KV)
            i, j = torch.arange(T, device=dev)[:, None], torch.arange(T, device=dev)[None, :]
            allowed = (j <= i) & (j > i - W) if W else (j <= i)
            qr, kr, vr = (x.clone().requires_grad_(True) for x in (q, k, v))
            ref = []
            for h in range(H):  # fp32 reference, one head at a time
                sc = (qr[0, h] @ kr[0, h // (H // KV)].T).masked_fill(~allowed, float("-inf"))
                ref.append(torch.softmax(sc, -1) @ vr[0, h // (H // KV)])
            ref = torch.stack(ref, 1)  # [T, H, D]
            go = torch.randn_like(ref)
            (ref * go).sum().backward()
            gref = [x.grad for x in (qr, kr, vr)]
            for variant in ("lc_bias", "lc_bool", "hf_dense"):
                qx, kx, vx = (x.to(dt).clone().requires_grad_(True) for x in (q, k, v))
                LC["attn"] = variant != "hf_dense"
                fix_attention.MASK = "bool" if variant == "lc_bool" else "bias"
                mask = None if variant != "hf_dense" or not W else allowed[None, None]
                o, _ = fix_attention.lc_attention(mod, qx, kx, vx, mask, scaling=1.0, sliding_window=W)
                (o.float() * go[None]).sum().backward()
                err = float((o[0].float() - ref).norm() / ref.norm())
                cos = [float(torch.nn.functional.cosine_similarity(a.grad.float().flatten(), b.flatten(), 0))
                       for a, b in zip((qx, kx, vx), gref)]
                rows.append({"kind": kind, "T": T, "variant": variant, "fwd_rel_err": round(err, 6),
                             "grad_cos_qkv": [round(c, 6) for c in cos]})
            del q, k, v, qr, kr, vr, ref, go, gref
    fix_attention.MASK = "bias"
    return rows


# ------------------------------------------------------------------ memory breakdown (allocator trace replay)
_KEEP = ("trainkit/", "transformers/", "peft/", "bitsandbytes/", "torch/utils/checkpoint", "torch/distributed/fsdp",
         "torch/autograd/graph", "torch/nn/functional")


def _frame_key(frames):
    for f in frames:  # innermost first
        fn = f.get("filename", "")
        if any(k in fn for k in _KEEP):
            short = fn.split("site-packages/")[-1].split("working/")[-1]
            return f"{short}:{f.get('line')}:{f.get('name')}"
    return "other"


def _breakdown(base_bytes, top=18):
    """Replay this device's allocator trace: the moment of peak live bytes and what was live then, by allocating
    frame. base_bytes = memory allocated when recording started (weights etc., not in the trace)."""
    snap = torch.cuda.memory._snapshot()
    trace = snap["device_traces"][torch.cuda.current_device()]
    live, cur, peak, peak_i = {}, 0, 0, -1
    for i, e in enumerate(trace):
        a = e.get("action")
        if a == "alloc":
            live[e["addr"]] = e["size"]
            cur += e["size"]
            if cur > peak:
                peak, peak_i = cur, i
        elif a in ("free_requested", "free_completed") and e["addr"] in live:
            cur -= live.pop(e["addr"])
    live, by = {}, {}
    for e in trace[:peak_i + 1]:
        a = e.get("action")
        if a == "alloc":
            live[e["addr"]] = (e["size"], _frame_key(e.get("frames", [])))
        elif a in ("free_requested", "free_completed"):
            live.pop(e["addr"], None)
    for size, k in live.values():
        by[k] = by.get(k, 0) + size
    rows = sorted(by.items(), key=lambda kv: -kv[1])[:top]
    return {"base_gib": round(base_bytes / 2 ** 30, 2), "trace_peak_gib": round(peak / 2 ** 30, 2),
            "total_peak_gib": round((base_bytes + peak) / 2 ** 30, 2), "trace_events": len(trace),
            "top": [[k, round(v / 2 ** 30, 3)] for k, v in rows]}


# ------------------------------------------------------------------ probe
def probe(tr, smoke):
    out = {"smoke": smoke, "flags": list(FLAGS)}
    path = OUT_DIR / "lc_probe.json"
    dev = tr.dev

    def save():
        if tr.rank == 0:
            path.write_text(json.dumps(out, indent=1))

    def reset():
        gc.collect()
        if not smoke:
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

    def peak():
        return 0 if smoke else round(torch.cuda.max_memory_allocated() / 2 ** 30, 2)

    if smoke:
        out["R_real_width"] = _real_width_diag(tr)
        save()
        out["U_attention"] = _attn_unit(dev, True)
        save()
        _smoke(tr, out, save)
        out["done"] = True
        save()
        return

    # LoRA B ~ N(0, 1e-3) (A keeps PEFT's kaiming init, identical on all ranks): BA*alpha/r ~ 1e-4 per weight entry,
    # ~0.5% of the base weights, so LoRA shapes the forward and both A and B get gradients
    _randomize(tr.fm.named_parameters(), std=1e-3, only="lora_B")

    tests = tr.s.lc_tests
    if "U" in tests:
        out["U_attention"] = _attn_unit(dev, False)  # before anything else: decides whether B/C below mean anything
        for r in out["U_attention"]:
            tr.log(event="lc_U", **r)
        save()

    def save_rank(key, row):  # every rank keeps its own copy: an OOM on rank r > 0 can hang rank 0 before it saves
        (OUT_DIR / f"lc_probe_rank{tr.rank}.json").write_text(json.dumps({**rank_rows, key: row}, indent=1))
        rank_rows[key] = row
    rank_rows = {}

    # D. rounding-only control (v3: attn on vs off LoRA-grad cosine 0.097 at 4k, although every attention path matched
    # fp32 in isolation and the real-width CPU model differed only at fp32 rounding level). Same math, three reduction
    # orders: HF dense mask, our blocks of 1024 queries, our blocks of 512. If 1024 vs 512 is as far apart as on vs
    # off, the 4k LoRA gradient is dominated by bf16 rounding and cosine cannot judge the attention fix.
    if "D" in tests:
        ids, lab, wts = _seq(4096, tr.rank, dev)
        norm = float(wts.sum())
        got, out["D_rounding"] = {}, []
        # v6: b512 vs b1024 cosine -0.53. Control: HF's own path with every attention output perturbed by up to one
        # bf16 ulp (2^-8 relative, two seeds). If HF vs HF + ulp noise is as far apart, any bf16 path is as good as HF's
        for name, block, noise in (("off", None, None), ("attn_b1024", None, None), ("attn_b512", 512, None),
                                   ("off_again", None, None), ("off_noise1", None, (2 ** -8, 1)),
                                   ("off_noise2", None, (2 ** -8, 2))):
            set_flags(tr.fm, attn=name.startswith("attn"))
            fix_attention.BLOCK, fix_attention.NOISE = block, noise
            reset()
            loss = _fwd_bwd(tr, ids, lab, wts, norm, smoke)
            got[name] = (loss, _grads(tr))
        fix_attention.BLOCK = fix_attention.NOISE = None
        for a, b in (("attn_b1024", "off"), ("attn_b512", "off"), ("attn_b512", "attn_b1024"), ("off_again", "off"),
                     ("off_noise1", "off"), ("off_noise2", "off"), ("off_noise2", "off_noise1")):
            row = {"pair": f"{a} vs {b}", "loss_a": round(got[a][0], 7), "loss_b": round(got[b][0], 7),
                   "loss_abs_diff": round(abs(got[a][0] - got[b][0]), 7), **_compare(got[a][1], got[b][1])}
            out["D_rounding"].append(row)
            tr.log(event="lc_D", **row)
        del got
        save()

    # A. StreamCE exactness on real hidden states; P_CHUNK 512 so the position chunks are exercised too (the CPU smoke
    # had one vocab tile and one chunk). Reference: full-logit autograd 512 positions at a time (2 GB fp32 per chunk)
    if "A" not in tests:
        out["done"] = True
        save()
        return
    reset()
    ids, lab, wts = _seq(2049, tr.rank, dev, tail=True)  # ends at the episode's last trained token (v3: the tail was context)
    set_flags(tr.fm, **{k: True for k in FLAGS})
    with torch.no_grad():
        tr.fm.eval()
        h = tr.fm(ids, lab, wts, hidden=True)  # through the FSDP root (root-owned params are only valid in there)
        tr.fm.train()
    head, cap = tr.fm.module.head[0], tr.fm.module.cap
    norm = float(wts.sum())
    hr = h.detach().clone().requires_grad_(True)
    chunk0, fix_head.P_CHUNK = fix_head.P_CHUNK, 512
    ls = StreamCE.apply(hr, head, lab.clamp(min=0), wts, cap, norm)
    fix_head.P_CHUNK = chunk0
    ls.backward()
    hf = h.detach().clone().requires_grad_(True)
    wg, lf = head.to(dev), 0.0
    for a in range(0, hf.shape[0], 512):
        part = reference_ce(hf[a:a + 512], wg, lab[a:a + 512].clamp(min=0), wts[a:a + 512], cap, norm)
        part.backward()
        lf += float(part.detach())
    del wg
    out["A_streamce"] = {"loss_stream": round(float(ls), 6), "loss_ref": round(lf, 6),
                         "abs_diff": round(abs(float(ls) - lf), 7), "peak_gb": peak(), **_compare([hr.grad.float()], [hf.grad.float()]),
                         "n_trained": int((wts > 0).sum())}
    tr.log(event="lc_A", **out["A_streamce"])
    del h, hr, hf, ls, lf
    save()

    if "B" not in tests:
        out["done"] = True
        save()
        return
    # B. exactness of each runtime fix at T=4096 (real tokens, one episode per rank)
    T = 4096
    ids, lab, wts = _seq(T, tr.rank, dev)
    norm = float(wts.sum())
    runs = [("off", {}), ("off_again", {})] + [(k, {k: True}) for k in FLAGS] + [("all", {k: True for k in FLAGS})]
    ref, out["B_grad"] = None, []
    for name, on in runs:
        set_flags(tr.fm, **on)
        reset()
        t = time.perf_counter()
        loss = _fwd_bwd(tr, ids, lab, wts, norm, smoke)
        row = {"run": name, "loss": round(loss, 6), "sec": round(time.perf_counter() - t, 1), "peak_gb": peak()}
        g = _grads(tr)
        if ref is None:
            ref, ref_loss = g, loss
        else:
            row.update(_compare(g, ref), loss_abs_diff=round(abs(loss - ref_loss), 7))
        out["B_grad"].append(row)
        save_rank(f"B_{name}", row)
        tr.log(event="lc_B", **row)
        save()
    del ref, g

    if "C" not in tests:
        out["done"] = True
        save()
        return
    # C. memory / time scan, all fixes on
    set_flags(tr.fm, **{k: True for k in FLAGS})
    out["C_scan"] = []
    last_ok = None
    for T in (28672, 32768):  # 8k: 8.24 GiB, 24k: 14.95 GiB peak (v2)
        ids, lab, wts = _seq(T, tr.rank, dev)
        reset()
        base = torch.cuda.memory_allocated()
        torch.cuda.memory._record_memory_history(max_entries=400000, stacks="python")
        row = {"T": T, "real_tokens": bool(_episodes())}
        t = time.perf_counter()
        try:
            loss = _fwd_bwd(tr, ids, lab, wts, float(wts.sum()), smoke)
            torch.cuda.synchronize()
            sec = time.perf_counter() - t
            row.update(ok=True, loss=round(loss, 4), sec=round(sec, 1), tok_s_rank=round(T / sec, 1), peak_gb=peak(),
                       reserved_gb=round(torch.cuda.max_memory_reserved() / 2 ** 30, 2))
            last_ok = T
        except torch.cuda.OutOfMemoryError as e:
            row.update(ok=False, error=f"OOM: {str(e)[:240]}", peak_gb=peak())
        try:
            row["breakdown"] = _breakdown(base)
        except Exception as e:  # the breakdown must never cost the result
            row["breakdown_error"] = repr(e)[:200]
        torch.cuda.memory._record_memory_history(enabled=None)
        flag = torch.tensor([0.0 if row["ok"] else 1.0], device=dev)
        dist.all_reduce(flag, op=dist.ReduceOp.MAX)
        row["any_rank_failed"] = bool(float(flag))
        row["host_avail_after_fwd_gb"] = HOST.get("avail_after_fwd_gb")
        out["C_scan"].append(row)
        save_rank(f"C_{T}", row)
        tr.log(event="lc_C", **{k: v for k, v in row.items() if k != "breakdown"})
        save()
        if row["any_rank_failed"]:
            break
    # one optimizer step on the last successful gradients (the trainer's real update path)
    if last_ok and not out["C_scan"][-1]["any_rank_failed"]:
        opt = ShardedAdamW(tr.lora, lr=1e-5)
        reset()
        t = time.perf_counter()
        gn = opt.step(1.0)
        torch.cuda.synchronize()
        out["C_opt_step"] = {"after_T": last_ok, "grad_norm": round(float(gn), 4), "sec": round(time.perf_counter() - t, 2),
                             "peak_gb": peak()}
        tr.log(event="lc_opt", **out["C_opt_step"])
    out["done"] = True
    save()


def _smoke(tr, out, save):
    """CPU fp32: new stack (FSDP, 2 gloo ranks) vs plain HF autograd on an unpatched copy, then a tiny train run."""
    from .model import add_lora, load_base
    dev, vocab = tr.dev, tr.vocab
    ref = add_lora(load_base(tr.model_path, tr.rank, True), tr.s.rank, tr.s.alpha, torch.float32)
    _randomize(ref.named_parameters())
    _randomize(tr.fm.named_parameters())
    ref_lora = {_key(n): p for n, p in ref.named_parameters() if "lora_" in n}
    name_of = {id(p): _key(n) for n, p in tr.fm.named_parameters() if "lora_" in n}
    mine = [name_of[id(p)] for p in tr.lora]  # tr.lora order (what _grads and ShardedAdamW use)
    assert sorted(mine) == sorted(ref_lora), "LoRA tensors differ between the FSDP model and the reference"
    T = 100  # > 6 sliding windows of 16 and > 3 MLP tiles of 32
    seqs = [_seq(T, k, dev, vocab) for k in range(tr.world)]
    norm = sum(float(w.sum()) for _, _, w in seqs)

    # reference: every rank computes all ranks' sequences on the unpatched model (flags off -> HF paths)
    for k in FLAGS:
        LC[k] = False
    ref.train()
    rlm = ref.base_model.model.model.language_model
    head = ref.base_model.model.get_output_embeddings().weight
    cap = tr.fm.module.cap

    def reference():
        for p in ref_lora.values():
            p.grad = None
        total = 0.0
        for ids, lab, wts in seqs:
            h = rlm(input_ids=ids, use_cache=False).last_hidden_state[0][:-1]
            loss = reference_ce(h, head, lab.clamp(min=0), wts, cap, norm)
            loss.backward()
            total += float(loss)
        return total, [ref_lora[k].grad.detach().float().clone() for k in mine]
    rloss, rgrad = reference()

    def new_stack(**on):
        set_flags(tr.fm, **on)
        ids, lab, wts = seqs[tr.rank]
        loss = _fwd_bwd(tr, ids, lab, wts, norm, True)
        g = [x.clone() for x in _grads(tr)]
        for x in g:
            dist.all_reduce(x)  # what ShardedAdamW's reduce-scatter sums
        tot = torch.tensor([loss], dtype=torch.float64)
        dist.all_reduce(tot)
        return float(tot), g

    out["S_exact"] = []
    for name, on in [("all", {k: True for k in FLAGS})] + [(f"all_but_{k}", {j: j != k for j in FLAGS}) for k in FLAGS] + \
                    [("none", {})]:
        loss, g = new_stack(**on)
        row = {"run": name, "loss": round(loss, 7), "loss_ref": round(rloss, 7), "loss_abs_diff": round(abs(loss - rloss), 8),
               **_compare(g, rgrad, reduce=False)}
        row["pass"] = row["cosine"] > 0.99999 and row["worst_tensor_rel"] < 1e-3 and row["loss_abs_diff"] < 1e-4
        out["S_exact"].append(row)
        tr.log(event="lc_S", **row)
        save()

    # the L4 noise control (test D) must perturb the forward consistently under checkpoint recompute: the new stack
    # with noise must match the reference with the same noise exactly, twice; noisy vs clean is reported for scale
    fix_attention.NOISE = (2 ** -8, 1)
    for k in FLAGS:
        LC[k] = False
    nloss, ngrad = reference()
    out["S_noise"] = []
    for run in ("noise_vs_noisy_ref", "noise_again_vs_noisy_ref"):
        loss, g = new_stack(**{k: True for k in FLAGS})
        row = {"run": run, "loss": round(loss, 7), "loss_ref": round(nloss, 7), "loss_abs_diff": round(abs(loss - nloss), 8),
               **_compare(g, ngrad, reduce=False)}
        row["pass"] = row["cosine"] > 0.99999 and row["worst_tensor_rel"] < 1e-3 and row["loss_abs_diff"] < 1e-4
        out["S_noise"].append(row)
        tr.log(event="lc_S_noise", **row)
    fix_attention.NOISE = None
    out["S_noise"].append({"run": "noisy_ref_vs_clean_ref", "loss_abs_diff": round(abs(nloss - rloss), 8),
                           **_compare(ngrad, rgrad, reduce=False)})
    save()

    # one optimizer step: ShardedAdamW (all fixes on) vs torch AdamW + clip on the reference, fed the same summed
    # gradients (Adam's first step is ~lr * sign(g), so near-zero grads that differ by noise would flip by 2 lr)
    _, g = new_stack(**{k: True for k in FLAGS})
    opt = ShardedAdamW(tr.lora, lr=1e-2)
    gn = opt.step(1.0)
    rparams = [ref_lora[k] for k in mine]
    for p, x in zip(rparams, g):
        p.grad = x.clone()
    rn = torch.nn.utils.clip_grad_norm_(rparams, 1.0)
    ropt = torch.optim.AdamW(rparams, lr=1e-2, weight_decay=0.0)
    ropt.step()
    diff = max(float((a.detach() - b.detach()).abs().max()) for a, b in zip(tr.lora, rparams))
    out["S_opt"] = {"grad_norm": round(float(gn), 6), "grad_norm_ref": round(float(rn), 6), "max_param_abs_diff": diff,
                    "pass": diff < 1e-5 and abs(float(gn) - float(rn)) < 1e-4 * max(1.0, float(rn))}
    tr.log(event="lc_S_opt", **out["S_opt"])
    save()
    del ref

    # tiny end-to-end train run on a synthetic rs2-format file
    f = OUT_DIR / "smoke_rs2.npz"
    if tr.rank == 0:
        rng = np.random.default_rng(0)
        lens = rng.integers(40, 120, size=14)
        ids = rng.integers(10, vocab, size=int(lens.sum())).astype(np.int32)
        w = (rng.random(int(lens.sum())) < 0.5).astype(np.float16) * 2
        np.savez(f, ids=ids, weight=w, ep_starts=np.concatenate([[0], np.cumsum(lens)]).astype(np.int64),
                 val=np.array([0] * 10 + [1] * 4, dtype=np.uint8))
    dist.barrier()
    data = TrainData(f, vocab=vocab, seed=0, max_len=110)
    tr.s.epochs, tr.s.grad_accum, tr.s.lr, tr.s.warmup_steps = 2.0, 2, 1e-3, 1
    tr.run(data)
    out["S_train"] = {**data.stats(), "plan_steps": len(data.plan(data.train_w, tr.world, 2, 2.0))}
    save()

### `trainkit/vablate.py`
LP2: HF-side ablation of the global layers' V delta (pre-registered in docs/experiments.md, Nov 1). Gemma-4's 10 global

In [ ]:
%%writefile /kaggle/working/trainkit/vablate.py
"""
LP2: HF-side ablation of the global layers' V delta (pre-registered in docs/experiments.md, Nov 1). Gemma-4's 10 global
layers have no v_proj: HF computes V = k_proj(x), so a k_proj LoRA moves K and V. LP1 found vLLM scores the RS2 val
episodes slightly better with the V delta absent (raw) than present (kv copy). Here, on the training stack (NF4 + HF +
LC1 fixes), the same comparison:
  full     T1's adapter as trained (K and V get the k_proj delta): must reproduce T1's end val 0.0666
  v_base   global layers' V = base k_proj(x) (LoRA skipped for V only; K keeps it): the vLLM 'raw' forward
  base     every lora_B zeroed: must reproduce T1's start val 0.1011
Weighted val loss pooled and per episode (as Trainer.validate). If v_base < full here too, the vLLM copy reproduces
HF and the V delta itself is mildly harmful on held-out episodes; if v_base > full, vLLM's kv arm is not HF's forward.
Smoke (CPU, tiny random model, random LoRA): the V patch is exact (full == v_base when the global k_proj delta is
zero) and changes the loss when it is not.
"""
import glob
import json

import torch
import torch.distributed as dist

from .config import INPUT, OUT_DIR

ADAPTER = "rs2_lora"


def norm_name(n):
    """FSDP-wrapped name -> export / checkpoint key prefix (as export.export_adapter)."""
    return n.replace("_fsdp_wrapped_module.", "").removeprefix("m.").replace(".default.", ".")


def global_attn(fm):
    """{key prefix of self_attn: (attention module, its k_proj LoRA module)} for every text attention layer without
    v_proj (the k_eq_v global layers)."""
    return {norm_name(n): (mod, mod.k_proj) for n, mod in fm.named_modules()
            if type(mod).__name__ == "Gemma4TextAttention" and mod.v_proj is None}


def set_v_base(pairs, on):
    """on: V = k_proj.base_layer(x) (no LoRA); off: back to HF's V = k_proj(x). object.__setattr__, so the base layer
    is not registered as a second submodule (FSDP already holds its weights in this layer's flat parameter)."""
    for attn, k in pairs.values():
        object.__setattr__(attn, "v_proj", k.base_layer if on else None)


def lora_named(tr):
    return {norm_name(n): p for n, p in tr.fm.named_parameters() if "lora_" in n}


def load_adapter(tr, smoke):
    """T1's export into the replicated LoRA tensors (smoke: random lora_B, std 0.05)."""
    named = lora_named(tr)
    if smoke:
        g = torch.Generator().manual_seed(1)
        for n, p in named.items():
            if "lora_B" in n:
                p.data.copy_((torch.randn(p.shape, generator=g) * 0.05).to(p.dtype))
        return {"source": "random lora_B (smoke)", "loaded": len(named)}
    from safetensors.torch import load_file
    found = sorted(glob.glob(f"{INPUT}/**/adapters/{ADAPTER}/adapter_model.safetensors", recursive=True))
    assert len(found) == 1, f"expected one attached adapters/{ADAPTER}/, found {found}"
    sd = load_file(found[0])
    missing = [n for n in named if n not in sd]
    extra = [k for k in sd if k not in named]
    assert not missing and all(".v_proj." in k for k in extra), (missing[:3], extra[:3])
    for n, p in named.items():
        assert sd[n].shape == p.shape, (n, sd[n].shape, p.shape)
        p.data.copy_(sd[n].to(p.device, p.dtype))
    return {"source": found[0], "loaded": len(named), "ignored_extra": len(extra)}


def per_episode(tr, data, tag):
    """Weighted val loss pooled + per episode; ranks split the episodes, equal counts per rank (FSDP)."""
    tr.fm.eval()
    mine = data.val_w[tr.rank::tr.world]
    n_real = len(mine)
    mine = mine + mine[:1] * (len(data.val_w[::tr.world]) - n_real)
    rows = torch.zeros(len(data.val_w), 2, dtype=torch.float64, device=tr.dev)
    with torch.no_grad():
        for k, w in enumerate(mine):
            ids, lab, wts = data.example(w, tr.dev)
            loss = float(tr.loss(ids, lab, wts, 1.0))  # sum_t w_t * ce_t
            if k < n_real:
                rows[data.val_w.index(w)] = torch.tensor([loss, float(wts.sum())], dtype=torch.float64)
    dist.all_reduce(rows)
    tot = rows.sum(0)
    out = {"loss": round(float(tot[0] / tot[1]), 5), "per_episode": [round(float(a / b), 5) for a, b in rows.tolist()]}
    tr.log(event="vablate", tag=tag, loss=out["loss"], windows=len(data.val_w))
    return out


def probe(tr, data, smoke):
    pairs = global_attn(tr.fm)
    report = {"global_layers": len(pairs), "adapter": load_adapter(tr, smoke), "arms": {}}
    assert pairs, "no k_eq_v layer found (expected v_proj None on global layers)"
    named = lora_named(tr)
    arms = report["arms"]
    if smoke:  # exactness: with the global k_proj delta zeroed, V is the base output either way
        glob_k = [f"{pre}.k_proj.lora_B.weight" for pre in pairs]
        kB = {n: named[n].data.clone() for n in glob_k}
        for n in glob_k:
            named[n].data.zero_()
        arms["kzero_full"] = per_episode(tr, data, "kzero_full")
        set_v_base(pairs, True)
        arms["kzero_v_base"] = per_episode(tr, data, "kzero_v_base")
        set_v_base(pairs, False)
        for n in glob_k:
            named[n].data.copy_(kB[n])
        report["smoke_exact_max_diff"] = max(abs(a - b) for a, b in zip(arms["kzero_full"]["per_episode"],
                                                                           arms["kzero_v_base"]["per_episode"]))
        report["smoke_global_k_zeroed"] = len(glob_k)
    arms["full"] = per_episode(tr, data, "full")
    set_v_base(pairs, True)
    arms["v_base"] = per_episode(tr, data, "v_base")
    set_v_base(pairs, False)
    arms["full_again"] = per_episode(tr, data, "full_again")  # the patch is undone cleanly
    for n, p in named.items():
        if "lora_B" in n:
            p.data.zero_()
    arms["base"] = per_episode(tr, data, "base")
    f, v = arms["full"]["per_episode"], arms["v_base"]["per_episode"]
    report["v_base_minus_full"] = round(arms["v_base"]["loss"] - arms["full"]["loss"], 5)
    report["v_base_better_episodes"] = sum(a < b for a, b in zip(v, f))
    report["full_again_max_diff"] = max(abs(a - b) for a, b in zip(f, arms["full_again"]["per_episode"]))
    if tr.rank == 0:
        (OUT_DIR / "vablate.json").write_text(json.dumps(report, indent=1))
    tr.log(event="vablate_done", **{k: v for k, v in report.items() if k not in ("arms", "adapter")})

### `trainkit/export.py`
Export: the (replicated) LoRA weights -> PEFT adapter in the format vLLM served in p0.

In [ ]:
%%writefile /kaggle/working/trainkit/export.py
"""Export: the (replicated) LoRA weights -> PEFT adapter in the format vLLM served in p0."""
import json
import os

import torch

from .config import TARGETS


def export_adapter(fm, layer_cls, out_dir, rank, r, alpha, base_path):
    """LoRA tensors are FSDP-ignored and identical on every rank (fix_lora), so rank 0 writes them directly. Keys are
    base_model.model.<checkpoint module>.lora_{A,B}.weight, e.g. ...model.language_model.layers.N.self_attn.q_proj."""
    from safetensors.torch import save_file
    if rank != 0:
        return None
    tensors = {}
    for n, p in fm.named_parameters():
        if "lora_" in n:
            key = n.replace("_fsdp_wrapped_module.", "").removeprefix("m.").replace(".default.", ".")
            tensors[key] = p.detach().to("cpu", torch.bfloat16).clone()
    # k_eq_v global layers: no v_proj LoRA, so vLLM serves V without the k_proj delta (HF trains with it). LP1/LP2:
    # that scores 0.3% better on held-out episodes than copying the delta to V (evalkit/kvdup.py, opt-in)
    os.makedirs(out_dir, exist_ok=True)
    save_file(tensors, os.path.join(out_dir, "adapter_model.safetensors"))
    with open(os.path.join(out_dir, "adapter_config.json"), "w") as f:
        json.dump({"peft_type": "LORA", "task_type": "CAUSAL_LM", "r": r, "lora_alpha": alpha, "lora_dropout": 0.0,
                   "bias": "none", "fan_in_fan_out": False, "target_modules": list(TARGETS),
                   "base_model_name_or_path": base_path, "inference_mode": True}, f, indent=1)
    mods = {k.removeprefix("base_model.model.").rsplit(".lora_", 1)[0] for k in tensors}
    return {"dir": out_dir, "n_modules": len(mods), "n_tensors": len(tensors), "key_example": next(iter(tensors)),
            "shapes_ok": all(v.ndim == 2 and min(v.shape) == r for v in tensors.values()),
            "mb": round(os.path.getsize(os.path.join(out_dir, "adapter_model.safetensors")) / 1e6, 1)}

### `trainkit/worker.py`
Torchrun entry point (one process per GPU): python -m torch.distributed.run --nproc_per_node 4 -m trainkit.worker MODEL

In [ ]:
%%writefile /kaggle/working/trainkit/worker.py
"""
Torchrun entry point (one process per GPU): python -m torch.distributed.run --nproc_per_node 4 -m trainkit.worker MODEL
Data parallel: each rank runs one window per micro-step; the loss is divided by the step's total label weight (all
ranks and micro-steps), LoRA grads are summed across ranks by ShardedAdamW (fix_lora), which steps every grad_accum
micro-steps with linear warmup then cosine decay. Validation loss (held-out tasks) at the start, every
val_every steps and at the end. Rank 0 writes metrics.jsonl and the adapter(s).
"""
import ctypes
import datetime
import gc
import json
import math
import sys
import threading
import time

import torch
import torch.distributed as dist

from .config import OUT_DIR, TrainSettings
from .data import TrainData
from .export import export_adapter
from .fix_lora import ShardedAdamW
from .model import add_lora, load_base, lora_modules, wrap_fsdp


def host_gb():
    """(this process's RSS, node MemAvailable) in GB, from /proc."""
    def kb(path, key):
        with open(path) as f:
            return int(next(l for l in f if l.startswith(key)).split()[1]) / 2 ** 20
    return round(kb("/proc/self/status", "VmRSS"), 1), round(kb("/proc/meminfo", "MemAvailable"), 1)


def release_heap():
    """Return freed heap to the OS: loading 62 GB of bf16 shards leaves glibc holding freed CPU memory (LC1 v4: a rank
    was SIGKILLed for host RAM 50 s after load, at the first 4k step)."""
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except OSError:
        pass


class HostWatch(threading.Thread):
    """Samples RSS / MemAvailable every 5 s into OUT_DIR/host_rank{r}.json (written each sample, so it survives a
    SIGKILL): the minimum MemAvailable, the maximum RSS and the last 120 samples."""

    def __init__(self, rank, t0):
        super().__init__(daemon=True)
        self.path, self.t0 = OUT_DIR / f"host_rank{rank}.json", t0
        self.min_avail, self.max_rss, self.tail = 1e9, 0.0, []

    def run(self):
        while True:
            rss, avail = host_gb()
            self.min_avail, self.max_rss = min(self.min_avail, avail), max(self.max_rss, rss)
            self.tail = (self.tail + [[round((time.time() - self.t0) / 60, 2), rss, avail]])[-120:]
            try:
                self.path.write_text(json.dumps({"min_avail_gb": self.min_avail, "max_rss_gb": self.max_rss,
                                                 "tail_min_rss_avail": self.tail}))
            except OSError:
                pass
            time.sleep(5)


class Trainer:
    def __init__(self, s, model_path):
        self.s, self.model_path = s, model_path
        dist.init_process_group("gloo" if s.smoke else "nccl", timeout=datetime.timedelta(minutes=10))
        self.rank, self.world = dist.get_rank(), dist.get_world_size()
        if s.smoke:
            self.dev, self.dt = torch.device("cpu"), torch.float32
        else:
            torch.cuda.set_device(self.rank)
            self.dev, self.dt = torch.device("cuda", self.rank), torch.bfloat16
            # flash caps head_dim at 256 and math materialises T x T: force the memory-efficient kernel
            torch.backends.cuda.enable_flash_sdp(False)
            torch.backends.cuda.enable_math_sdp(False)
            torch.backends.cuda.enable_cudnn_sdp(False)
            torch.backends.cuda.enable_mem_efficient_sdp(True)
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        self.host = None
        if not s.smoke:
            self.host = HostWatch(self.rank, s.t0)
            self.host.start()
        self.metrics = open(OUT_DIR / "metrics.jsonl", "a") if self.rank == 0 else None

    def log(self, **kw):
        kw["t_min"] = round((time.time() - self.s.t0) / 60, 1)
        print(f"rank{self.rank}", json.dumps(kw), flush=True)
        if self.metrics:
            self.metrics.write(json.dumps(kw) + "\n")
            self.metrics.flush()

    def build(self):
        t = time.time()
        m = load_base(self.model_path, self.rank, self.s.smoke)
        load_s = round(time.time() - t)
        m = add_lora(m, self.s.rank, self.s.alpha, self.dt)
        n_lora = lora_modules(m)
        self.fm, self.layer_cls, self.lora = wrap_fsdp(m, self.s.smoke, 32 if self.s.smoke else self.s.mlp_tile)  # smoke: T~100
        self.fm.train()  # from_pretrained returns eval mode, where HF skips gradient checkpointing (LC1 v4/v5: a 4k
        # step then kept every layer's activations and dequantized weights, ~40 GB/rank of host RAM via save_on_cpu)
        if not self.s.smoke:
            torch.cuda.empty_cache()
        self.vocab = m.config.get_text_config().vocab_size
        rss0, _ = host_gb()
        release_heap()
        rss1, avail = host_gb()
        self.log(event="model", load_s=load_s, lora_modules=n_lora, lora_params=sum(p.numel() for p in self.lora),
                 gpu_gb=0 if self.s.smoke else round(torch.cuda.memory_allocated() / 2 ** 30, 2),
                 rss_gb_before_trim=rss0, rss_gb=rss1, node_avail_gb=avail)

    def loss(self, ids, lab, wts, norm):
        """sum_t wts_t * ce_t / norm for one sequence; checkpoint inputs of long sequences live in pinned host RAM."""
        if ids.shape[1] >= self.s.offload_from:
            with torch.autograd.graph.save_on_cpu(pin_memory=False):  # pinned blocks round up: host OOM at 28k (LC1 v2)
                return self.fm(ids, lab, wts, norm)
        return self.fm(ids, lab, wts, norm)

    def val_totals(self, data, ws):
        """-> [weighted loss sum, weight sum] over the windows ws, all ranks."""
        tot = torch.zeros(2, device=self.dev, dtype=torch.float64)
        n_real = len(ws[self.rank::self.world])
        mine = ws[self.rank::self.world] + ws[:1] * (len(ws[::self.world]) - n_real)  # equal counts per rank (FSDP); repeat, weight 0
        with torch.no_grad():
            for k, w in enumerate(mine):
                ids, lab, wts = data.example(w, self.dev)
                real = k < n_real
                loss = self.loss(ids, lab, wts, 1.0)
                tot += torch.tensor([float(loss) * real, float(wts.sum()) * real], device=self.dev, dtype=torch.float64)
        dist.all_reduce(tot)
        return tot

    def validate(self, data, tag):
        """Weighted mean loss on the held-out tasks (every trained token counts the same); ranks split the windows.
        With corrected held-out episodes (rs4) the on-policy and corrected sets are also reported separately."""
        if not data.val_w:
            return None
        self.fm.eval()
        sets = {k: ws for k, ws in data.val_sets.items() if ws} or {"all": data.val_w}
        tot = torch.zeros(2, device=self.dev, dtype=torch.float64)
        for name, ws in sets.items():
            t = self.val_totals(data, ws)
            tot += t
            if name != "all":
                self.log(event="val_set", tag=tag, set=name, loss=round(float(t[0] / max(1e-6, float(t[1]))), 4), windows=len(ws))
        self.fm.train()
        v = float(tot[0] / max(1e-6, float(tot[1])))
        self.log(event="val", tag=tag, loss=round(v, 4), windows=len(data.val_w))
        return v

    def export(self, tag):
        out = OUT_DIR / ("adapter_" + tag)
        info = export_adapter(self.fm, self.layer_cls, str(out), self.rank, self.s.rank, self.s.alpha, self.model_path)
        dist.barrier()
        if info:
            self.log(event="export", tag=tag, **info)

    def lr_at(self, step, total):
        if step < self.s.warmup_steps:
            return self.s.lr * (step + 1) / self.s.warmup_steps
        p = (step - self.s.warmup_steps) / max(1, total - self.s.warmup_steps)
        return self.s.lr * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))

    def run(self, data):
        plan = data.plan(data.train_w, self.world, self.s.grad_accum, self.s.epochs)
        total = len(plan)
        self.log(event="plan", optimizer_steps=total, micro_steps=total * self.s.grad_accum, **data.stats())
        opt = ShardedAdamW(self.lora, lr=self.s.lr)
        self.validate(data, "start")
        for step, micro in enumerate(plan):
            stop = torch.tensor([1.0 if self.s.hours_left() < 0.4 else 0.0], device=self.dev)
            dist.all_reduce(stop, op=dist.ReduceOp.MAX)  # every rank stops at the same step
            if float(stop):
                self.log(event="budget_stop", optimizer_step=step)
                break
            t0, norm, tok, lsum = time.perf_counter(), data.step_norm(micro), 0, 0.0
            for windows in micro:
                ids, lab, wts = data.example(windows[self.rank], self.dev)
                loss = self.loss(ids, lab, wts, norm)
                loss.backward()
                lsum += float(loss.detach())
                tok += ids.shape[1]
            for g in opt.param_groups:
                g["lr"] = self.lr_at(step, total)
            gn = opt.step(1.0)
            opt.zero_grad()
            if not self.s.smoke:
                torch.cuda.synchronize(self.dev)
            dt = time.perf_counter() - t0
            local_peak = 0 if self.s.smoke else torch.cuda.max_memory_allocated() / 2 ** 30
            red = torch.tensor([tok, lsum, 0.0, 0.0], device=self.dev, dtype=torch.float64)
            dist.all_reduce(red)  # tokens and the step's weighted mean loss (each rank's part is already / norm)
            mx = torch.tensor([dt, local_peak], device=self.dev, dtype=torch.float64)
            dist.all_reduce(mx, op=dist.ReduceOp.MAX)
            self.log(event="step", step=step + 1, of=total, loss=round(float(red[1]), 4), lr=round(self.lr_at(step, total), 7),
                     grad_norm=round(float(gn), 3), tokens=int(red[0]), elapsed_s=round(float(mx[0]), 3),
                     tok_per_s=round(float(red[0]) / float(mx[0]), 1), peak_gb=round(float(mx[1]), 2),
                     max_len=max(data.length(w) for ws in micro for w in ws))
            last = step + 1 == total  # the end-of-run validation and export follow anyway
            if self.s.val_every and (step + 1) % self.s.val_every == 0 and not last:
                self.validate(data, f"step{step + 1}")
            if self.s.save_every and (step + 1) % self.s.save_every == 0 and not last:
                self.export(f"step{step + 1}")
        self.validate(data, "end")
        self.export("final")

    def close(self):
        dist.barrier()
        dist.destroy_process_group()


def main():
    s = TrainSettings.load()
    tr = Trainer(s, sys.argv[1])
    tr.build()
    if s.prefix_probe or s.lc_probe:
        if s.prefix_probe:
            from .prefix import probe
        else:
            from .lcprobe import probe
        probe(tr, s.smoke)
        tr.close()
        return
    data = TrainData(sys.argv[2], s.max_windows, vocab=tr.vocab if s.smoke else None, seed=s.seed, max_len=s.max_len,
                     corr=s.corr, corr_share=s.corr_share)
    if s.vablate:
        from .vablate import probe
        probe(tr, data, s.smoke)
    else:
        tr.run(data)
    tr.close()


if __name__ == "__main__":
    main()

### `trainkit/launcher.py`
Launcher: torchrun over the worker inside the scorer venv, with per-rank logs kept for post-mortems.

In [ ]:
%%writefile /kaggle/working/trainkit/launcher.py
"""Launcher: torchrun over the worker inside the scorer venv, with per-rank logs kept for post-mortems."""
import subprocess
import sys

from .config import OUT_DIR, WORK


def launch(model_path, data_path, nproc, timeout_s):
    logs = OUT_DIR / "rank_logs"
    logs.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "-m", "torch.distributed.run", "--nproc_per_node", str(nproc), "--log-dir", str(logs),
           "--redirects", "3", "--tee", "3", "-m", "trainkit.worker", str(model_path), str(data_path)]
    print("$", " ".join(cmd), flush=True)
    env = {"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "PYTHONPATH": str(WORK)}
    import os
    proc = subprocess.Popen(cmd, cwd=WORK, env={**os.environ, **env}, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        return proc.wait(timeout=timeout_s)
    except subprocess.TimeoutExpired:
        proc.kill()
        return -9


def rank_tails(n=1500):
    """Last lines of each rank's stderr (a crashing rank's own traceback, which torchrun hides)."""
    out = {}
    for f in sorted((OUT_DIR / "rank_logs").glob("**/stderr.log")):
        t = f.read_text(errors="ignore")[-n:]
        if t.strip():
            out[str(f.relative_to(OUT_DIR))] = t
    return out

### `trainkit/main.py`
Everything after the env step, inside the scorer venv: python -m trainkit.main (cwd /kaggle/working).

In [ ]:
%%writefile /kaggle/working/trainkit/main.py
"""Everything after the env step, inside the scorer venv: python -m trainkit.main (cwd /kaggle/working)."""
import glob
import json
import shutil

from .config import DATA_FILE, INPUT, OUT_DIR, SMOKE_CONFIG, WORK, TrainSettings
from .launcher import launch, rank_tails


def model_path(smoke):
    if smoke:  # tiny random Gemma-4 from the real config (same architecture, 6 layers)
        import torch
        from transformers import AutoConfig, AutoModelForImageTextToText
        real = json.loads(SMOKE_CONFIG.read_text())
        tc, n = real["text_config"], 6
        tc.update(hidden_size=64, intermediate_size=128, num_hidden_layers=n, num_attention_heads=4,
                  num_key_value_heads=2, num_global_key_value_heads=1, head_dim=16, global_head_dim=32, vocab_size=512,
                  vocab_size_per_layer_input=512, sliding_window=16,
                  layer_types=["sliding_attention"] * (n - 1) + ["full_attention"])
        real["vision_config"].update(hidden_size=32, intermediate_size=64, num_hidden_layers=1, num_attention_heads=2,
                                     num_key_value_heads=2, head_dim=16, global_head_dim=16)
        torch.manual_seed(0)
        cfg = AutoConfig.for_model(**{k: v for k, v in real.items() if k != "transformers_version"})
        path = WORK / "tiny_model"
        AutoModelForImageTextToText.from_config(cfg).save_pretrained(path)
        return path
    cfgs = sorted(p for p in glob.glob(f"{INPUT}/**/config.json", recursive=True)
                  if "31b" in p.lower() and "unquantized" in p.lower())
    assert cfgs, "attach model google/gemma-4/Transformers/gemma-4-31b-it-qat-q4_0-unquantized/2"
    return cfgs[0].rsplit("/", 1)[0]


def data_path():
    if DATA_FILE.exists():
        return DATA_FILE
    found = sorted(glob.glob(f"{INPUT}/**/train_*.npz", recursive=True)) + sorted(glob.glob(f"{INPUT}/**/rs[234].npz", recursive=True))
    assert found, "no training data: embed it (notebook inputs cell) or attach a dataset with train_*.npz / rs2.npz"
    return found[0]


def main():
    s = TrainSettings.load()
    mp, dp = model_path(s.smoke), ("none" if s.prefix_probe or s.lc_probe else data_path())
    print("model", mp, "| data", dp, "| settings", json.dumps(s.__dict__), flush=True)
    rc = launch(mp, dp, 2 if s.smoke else 4, timeout_s=int(max(0.2, s.hours_left() + 0.3) * 3600))
    report = {"rc": rc, "settings": s.__dict__}
    mfile = OUT_DIR / "metrics.jsonl"
    rows = [json.loads(l) for l in open(mfile)] if mfile.exists() else []
    report["val"] = [r for r in rows if r.get("event") == "val"]
    report["val_sets"] = [r for r in rows if r.get("event") == "val_set"]
    report["exports"] = [r for r in rows if r.get("event") == "export"]
    steps = [r for r in rows if r.get("event") == "step"]
    report["steps"] = {"n": len(steps), "first": steps[:3], "last": steps[-3:]}
    for probe in ("prefix_probe", "lc_probe", "vablate"):
        pp = OUT_DIR / f"{probe}.json"
        if pp.exists():
            report[probe] = json.loads(pp.read_text())
    if rc != 0:
        report["rank_tails"] = rank_tails()
    final = OUT_DIR / "adapter_final"
    if final.exists():  # ready to drop into a submission: adapters/<name>/
        dest = WORK / "adapters" / s.name
        shutil.rmtree(dest, ignore_errors=True)
        shutil.copytree(final, dest)
        report["adapter"] = str(dest)
    (WORK / "train_report.json").write_text(json.dumps(report, indent=1, default=str))
    print("TRAIN_REPORT", json.dumps({k: report[k] for k in ("rc", "val", "val_sets", "exports", "prefix_probe", "lc_probe", "vablate") if k in report}), flush=True)


if __name__ == "__main__":
    main()

## 6. Scorer environment
Restore the Py3.12 venv, upgrade changed wheels, add peft + bitsandbytes.

In [ ]:
sys.path.insert(0, "/kaggle/working")
from evalkit.envsetup import ScorerEnv
ScorerEnv().prepare_training(gpus=True)

## 7. Train
Runs inside the venv: torchrun over the GPUs (2 CPU ranks for the smoke); streamed here.

In [ ]:
import subprocess
proc = subprocess.Popen(["/tmp/scorer_host/bin/python", "-m", "trainkit.main"], cwd="/kaggle/working",
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
print("exit code", proc.wait())

## 8. Results

In [ ]:
print(open("/kaggle/working/train_report.json").read()[:20000])